# Stochastic SINDy comparison

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.4, Table 5, Appendix B.2.

Cross-validated stochastic SINDy (stepwise sparse regression, Kramers–Moyal targets, trajectory-grouped 5-fold CV) versus vanilla and Gaeta–Lie neural SDEs under a shared protocol.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Imports, global E5 configuration, deterministic seed utilities

In [ ]:
# =============================================================================
# Cell 1 — Imports, global E5 configuration, deterministic seed utilities
# =============================================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import hashlib
import json
import math
import os
import time
import warnings
from dataclasses import dataclass
from functools import partial
from itertools import combinations_with_replacement
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


CONFIG = {
    # -------------------------------------------------------------------------
    # Run control
    # -------------------------------------------------------------------------
    # "smoke": one tiny development run.
    # "single": one complete benchmark / regime / seed.
    # "full": all configured benchmarks, regimes, and seeds.
    "EXECUTION_MODE": "single",
    "RUN_E5_NOW": True,

    "ACTIVE_BENCHMARK": "ex_1_1",
    "ACTIVE_REGIME": "efficient",
    "ACTIVE_SEED": 11,

    "E5_BENCHMARKS": [
        "ex_1_1",
        "ex_1_3",
        "ex_3_1",
        "ex_3_2",
        "ex_20_3",
    ],
    "E5_REGIMES": ["efficient", "scarce"],
    "E5_SEEDS": [11, 22, 33, 44, 55],

    # Optional additional stress test. It is not needed to satisfy E5.
    "INCLUDE_OPTIONAL_50D": False,
    "OPTIONAL_50D_BENCHMARK": "ex_50_1",

    # -------------------------------------------------------------------------
    # Persistent output and resumption
    # -------------------------------------------------------------------------
    # For persistence across sessions, change this to a persistent directory
    # directory, for example:
    #   /path/to/persistent/storage/E5_results
    "RESULTS_DIR": "./e5_results",
    "RESUME_FROM_EXISTING_CSV": True,
    "SKIP_COMPLETED_CASES": True,

    # -------------------------------------------------------------------------
    # Data regimes
    # -------------------------------------------------------------------------
    # Manuscript efficient setting: N_traj = 2d.
    # E5 extends the scarce setting to every tested dimension using the same
    # 60% fraction as the submitted 50D comparison: 60 / 100 = 0.6.
    "TRAIN_FRACTION": 0.80,
    "SCARCE_FRACTION": 0.60,

    # -------------------------------------------------------------------------
    # Shared evaluation
    # -------------------------------------------------------------------------
    "N_EVAL_POINTS_BOX": 4000,
    "N_EVAL_POINTS_SUPPORT": 4000,
    "N_ROLLOUT_PATHS": 64,
    "ROLLOUT_STATE_ABS_LIMIT": 1.0e6,
    "ROLLOUT_DIVERGENCE_PENALTY": 1.0e12,
    "Q_FLOOR": 1.0e-8,
    "SIGMA_EPS": 1.0e-4,

    # -------------------------------------------------------------------------
    # Neural-SDE optimization
    # -------------------------------------------------------------------------
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2.0e-3,
    "BASELINE_PRINT_EVERY": 250,

    "SYM_DISCOVERY_LR": 1.0e-2,
    "SYM_INIT_SCALE": 0.05,
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.10,
    "LAMBDA_IND": 0.05,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.15,
    "CLOSURE_RCOND": 1.0e-5,
    "BASIS_EXP_CLIP": 8.0,

    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 1.0e-3,
    "STAGE_F_LAMBDA_SYM_DET": 0.05,

    # -------------------------------------------------------------------------
    # Boninsegna-style stochastic SINDy: Kramers--Moyal + CV-SSR
    # -------------------------------------------------------------------------
    "SINDY_CV_FOLDS": 5,
    "SINDY_LSTSQ_RCOND": 1.0e-10,
    "SINDY_FEATURE_SCALE_FLOOR": 1.0e-10,
    "SINDY_COEF_ZERO_TOL": 1.0e-10,

    # Primary elbow diagnostic. For model size n, compare CV(n-1)/CV(n).
    # A large ratio means that removing the nth term caused a sharp failure.
    "SINDY_CV_RATIO_THRESHOLD": 2.0,
    "SINDY_CV_NEAR_MIN_REL_TOL": 0.25,

    # Robust fallback when no unambiguous ratio elbow is present: choose the
    # smallest model whose mean CV error is within one standard error of the
    # minimum-CV model.
    "SINDY_USE_ONE_STANDARD_ERROR_FALLBACK": True,

    # Ordinary least squares is used, as in SSR. Feature columns are normalized
    # only to make coefficient magnitudes comparable during backward removal;
    # reported coefficients are transformed back to the physical library.
    "SINDY_PROTECT_CONSTANT": False,

    # Generic, non-oracle Fourier bank. Exact benchmark-specific terms are added
    # only to the oracle-enriched library.
    "SINDY_STANDARD_TIME_FREQUENCIES": [0.5, 1.0, 2.0, 3.0],
    "SINDY_TIME_STATE_PRODUCT_FREQUENCIES": [1.0],

    # -------------------------------------------------------------------------
    # Development-mode overrides
    # -------------------------------------------------------------------------
    "SMOKE_BASELINE_EPOCHS": 3,
    "SMOKE_SYM_EPOCHS": 2,
    "SMOKE_STAGE_F_EPOCHS": 2,
    "SMOKE_N_EVAL_POINTS_BOX": 64,
    "SMOKE_N_EVAL_POINTS_SUPPORT": 64,
    "SMOKE_N_ROLLOUT_PATHS": 4,
    "SMOKE_MAX_SINDY_FEATURES": 24,

    # -------------------------------------------------------------------------
    # Reporting
    # -------------------------------------------------------------------------
    "PRINT_TOP_SINDY_TERMS": 12,
}


METHOD_ORDER = [
    "Vanilla neural SDE",
    "Gaeta-Lie",
    "Stochastic SINDy (standard)",
    "Stochastic SINDy (oracle)",
]


def stable_seed(*parts):
    """Create a reproducible 31-bit seed from arbitrary labels."""
    payload = "||".join(str(part) for part in parts).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return int.from_bytes(digest[:8], byteorder="little", signed=False) % (2**31 - 1)


def block_until_ready_tree(tree):
    """Synchronize all JAX leaves before stopping a wall-clock timer."""
    leaves = jax.tree_util.tree_leaves(tree)
    for leaf in leaves:
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return tree


def as_1d_parameter(value, d, dtype=jnp.float32):
    arr = jnp.asarray(value, dtype=dtype)
    if arr.ndim == 0:
        arr = jnp.full((d,), arr, dtype=dtype)
    if arr.shape != (d,):
        raise ValueError(f"Expected scalar or shape {(d,)}, got {arr.shape}.")
    return arr


def regime_trajectory_count(d, regime):
    efficient = int(2 * d)
    if regime == "efficient":
        return efficient
    if regime == "scarce":
        return max(1, int(round(CONFIG["SCARCE_FRACTION"] * efficient)))
    raise ValueError(f"Unknown regime: {regime}")


def effective_count(name, regular_value, smoke_value):
    if CONFIG["EXECUTION_MODE"] == "smoke":
        return int(smoke_value)
    return int(regular_value)


RESULTS_DIR = Path(CONFIG["RESULTS_DIR"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LONG_RESULTS_PATH = RESULTS_DIR / "E5_results_long.csv"

print("E5 results directory:", RESULTS_DIR)
print("Execution mode:", CONFIG["EXECUTION_MODE"])


## Cell 2 — Exact manuscript benchmark registry for the primary E5 suite

In [ ]:
# =============================================================================
# Cell 2 — Exact manuscript benchmark registry for the primary E5 suite
# =============================================================================


def drift_ex_1_1(t, x):
    return -0.8 * x + 0.25 * jnp.sin(1.7 * t) + 0.10 * jnp.tanh(x)


def sigma_ex_1_1(t, x):
    return 0.35 + 0.08 * jnp.sin(t) ** 2 + 0.12 * jax.nn.sigmoid(x)


def drift_ex_1_3(t, x):
    return (
        -0.9 * x
        - 0.25 * jnp.sin(1.5 * t)
        + 0.16 * jnp.tanh(1.4 * x)
        + 0.08 * jnp.sin(t) * x
        - 0.04 * x**3
    )


def sigma_ex_1_3(t, x):
    return (
        0.34
        + 0.06 * jnp.cos(1.2 * t) ** 2
        + 0.055 * jnp.tanh(0.9 * x) ** 2
        + 0.06 * jax.nn.sigmoid(-x + 0.3 * jnp.sin(t))
    )


def drift_ex_3_1(t, x):
    x1, x2, x3 = x[..., 0], x[..., 1], x[..., 2]
    f1 = -0.55 * x1 + 0.90 * jnp.tanh(x2) - 0.35 * jnp.tanh(x3) + 0.10 * jnp.sin(t)
    f2 = -0.60 * x2 + 0.85 * jnp.tanh(x3) - 0.25 * jnp.tanh(x1)
    f3 = -0.50 * x3 + 0.80 * jnp.tanh(x1) - 0.30 * jnp.tanh(x2) + 0.10 * jnp.cos(1.3 * t)
    return jnp.stack([f1, f2, f3], axis=-1)


def sigma_ex_3_1(t, x):
    return 0.24 + 0.05 * jnp.sin(x) ** 2


def drift_ex_3_2(t, x):
    x1, x2, x3 = x[..., 0], x[..., 1], x[..., 2]
    forcing = 0.15 * jnp.sin(2.0 * jnp.pi * t)
    f1 = x1 - x1**3 + 0.35 * (x2 - x1) + 0.20 * (x3 - x1) + forcing
    f2 = x2 - x2**3 + 0.35 * (x1 - x2) + 0.25 * (x3 - x2)
    f3 = x3 - x3**3 + 0.25 * (x2 - x3) + 0.20 * (x1 - x3) - forcing
    return jnp.stack([f1, f2, f3], axis=-1)


def sigma_ex_3_2(t, x):
    base = jnp.asarray([0.22, 0.28, 0.25], dtype=x.dtype)
    amp = jnp.asarray([0.06, 0.05, 0.07], dtype=x.dtype)
    return base + amp * jax.nn.sigmoid(x)


def _ring_phase(d, dtype=jnp.float32):
    return 2.0 * jnp.pi * jnp.arange(d, dtype=dtype) / float(d)


def drift_ex_20_3(t, x):
    d = x.shape[-1]
    theta = _ring_phase(d, x.dtype)
    damping = 0.50 + 0.10 * jnp.sin(theta)
    return (
        -damping * x
        + 0.72 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.30 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.18 * jnp.tanh(jnp.roll(x, -2, axis=-1))
        + 0.08 * jnp.sin(t + theta)
    )


def sigma_ex_20_3(t, x):
    d = x.shape[-1]
    theta = _ring_phase(d, x.dtype)
    base = 0.24 + 0.020 * jnp.sin(theta + 0.5)
    amp = 0.045 + 0.010 * jnp.sin(theta + 1.1)
    return base + amp * jnp.sin(x) ** 2


def drift_ex_50_1(t, x):
    d = x.shape[-1]
    theta = _ring_phase(d, x.dtype)
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x
    return x - x**3 + 0.25 * lap + 0.08 * jnp.sin(2.0 * jnp.pi * t + theta)


def sigma_ex_50_1(t, x):
    d = x.shape[-1]
    theta = _ring_phase(d, x.dtype)
    return 0.18 + 0.02 * jnp.sin(theta) ** 2 + 0.04 * jax.nn.sigmoid(x)


BENCHMARKS = {
    "ex_1_1": {
        "display_name": "Ex 1.1",
        "description": "1D nonpolynomial drift and state-dependent diffusion.",
        "d": 1,
        "drift_fn": drift_ex_1_1,
        "sigma_fn": sigma_ex_1_1,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.7,
        "x_box": (-2.5, 2.5),
        "input_x_scale": 2.5,
        "hidden_dim": 64,
        "depth": 3,
        "baseline_epochs": 1500,
        "n_generators": 3,
        "sym_epochs": 2000,
        "sym_points": 512,
        "stage_f_epochs": 1200,
        "stage_f_extra_points": 1024,
        "basis_mu_list": [-2.0, -0.5, 0.5, 2.0],
        "sym_collocation": "uniform_box",
    },
    "ex_1_3": {
        "display_name": "Ex 1.3",
        "description": "1D mixed polynomial, trigonometric, tanh, and sigmoid SDE.",
        "d": 1,
        "drift_fn": drift_ex_1_3,
        "sigma_fn": sigma_ex_1_3,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.7,
        "x_box": (-2.5, 2.5),
        "input_x_scale": 2.5,
        "hidden_dim": 64,
        "depth": 3,
        "baseline_epochs": 1500,
        "n_generators": 3,
        "sym_epochs": 2000,
        "sym_points": 512,
        "stage_f_epochs": 1200,
        "stage_f_extra_points": 1024,
        "basis_mu_list": [-2.0, -0.5, 0.5, 2.0],
        "sym_collocation": "uniform_box",
    },
    "ex_3_1": {
        "display_name": "Ex 3.1",
        "description": "3D cyclic nonlinear reaction SDE.",
        "d": 3,
        "drift_fn": drift_ex_3_1,
        "sigma_fn": sigma_ex_3_1,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": (0.0, 0.0, 0.0),
        "x0_std": (1.0, 1.0, 1.0),
        "x_box": (-3.0, 3.0),
        "input_x_scale": 3.0,
        "hidden_dim": 96,
        "depth": 3,
        "baseline_epochs": 2000,
        "n_generators": 5,
        "sym_epochs": 3000,
        "sym_points": 512,
        "stage_f_epochs": 1200,
        "stage_f_extra_points": 512,
        "basis_mu_list": [-2.0, -0.5, 0.5, 2.0],
        "sym_collocation": "uniform_box",
    },
    "ex_3_2": {
        "display_name": "Ex 3.2",
        "description": "3D coupled double-well SDE.",
        "d": 3,
        "drift_fn": drift_ex_3_2,
        "sigma_fn": sigma_ex_3_2,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": (0.0, 0.0, 0.0),
        "x0_std": (0.75, 0.75, 0.75),
        "x_box": (-2.5, 2.5),
        "input_x_scale": 2.5,
        "hidden_dim": 96,
        "depth": 3,
        "baseline_epochs": 2000,
        "n_generators": 5,
        "sym_epochs": 3000,
        "sym_points": 512,
        "stage_f_epochs": 1200,
        "stage_f_extra_points": 512,
        "basis_mu_list": [-1.0, -0.5, 0.5, 1.0],
        "sym_collocation": "uniform_box",
    },
    "ex_20_3": {
        "display_name": "Ex 20.3",
        "description": "20D cyclic reaction ring.",
        "d": 20,
        "drift_fn": drift_ex_20_3,
        "sigma_fn": sigma_ex_20_3,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "x_box": (-3.0, 3.0),
        "input_x_scale": 3.0,
        "hidden_dim": 96,
        "depth": 3,
        "baseline_epochs": 2000,
        "n_generators": 5,
        "sym_epochs": 3000,
        "sym_points": 256,
        "stage_f_epochs": 1200,
        "stage_f_extra_points": 256,
        "basis_mu_list": [-2.0, -0.5, 0.5, 2.0],
        "sym_collocation": "uniform_box",
    },
    "ex_50_1": {
        "display_name": "Ex 50.1",
        "description": "Optional 50D ring-coupled double-well stress test.",
        "d": 50,
        "drift_fn": drift_ex_50_1,
        "sigma_fn": sigma_ex_50_1,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "x_box": (-2.5, 2.5),
        "input_x_scale": 2.5,
        "hidden_dim": 192,
        "depth": 3,
        "baseline_epochs": 2000,
        "n_generators": 50,
        "sym_epochs": 4000,
        "sym_points": 256,
        "stage_f_epochs": 800,
        "stage_f_extra_points": 128,
        "basis_mu_list": [-1.0, -0.5, 0.5, 1.0],
        "sym_collocation": "trajectory_jitter",
        "sym_jitter_std": 0.05,
    },
}


for benchmark_name, spec in BENCHMARKS.items():
    d = int(spec["d"])
    spec["x0_mean_array"] = as_1d_parameter(spec["x0_mean"], d)
    spec["x0_std_array"] = as_1d_parameter(spec["x0_std"], d)

print("Primary E5 benchmarks:", CONFIG["E5_BENCHMARKS"])



## Cell 3 — Nested trajectory pools, one-step transition tables, paired splits

In [ ]:
# =============================================================================
# Cell 3 — Nested trajectory pools, one-step transition tables, paired splits
# =============================================================================


def simulate_sde_paths(spec, n_trajectories, seed):
    """Euler--Maruyama simulation returning times and paths (N,T+1,d)."""
    d = int(spec["d"])
    dt = float(spec["dt"])
    n_steps = int(round(float(spec["t_final"]) / dt))
    n_trajectories = int(n_trajectories)

    key = jax.random.PRNGKey(int(seed))
    key_x0, key_noise = jax.random.split(key)

    x0_mean = spec["x0_mean_array"]
    x0_std = spec["x0_std_array"]

    x0 = x0_mean + x0_std * jax.random.normal(
        key_x0,
        (n_trajectories, d),
        dtype=jnp.float32,
    )
    eps = jax.random.normal(
        key_noise,
        (n_steps, n_trajectories, d),
        dtype=jnp.float32,
    )
    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(jnp.asarray(dt, dtype=jnp.float32))

    drift_fn = spec["drift_fn"]
    sigma_fn = spec["sigma_fn"]

    @jax.jit
    def run(x_initial, noises):
        def step(x, inp):
            t, noise = inp
            f = drift_fn(t, x)
            sigma = sigma_fn(t, x)
            x_next = x + f * dt + sigma * sqrt_dt * noise
            return x_next, x_next

        _, tail = jax.lax.scan(step, x_initial, (times[:-1], noises))
        return jnp.concatenate([x_initial[None, ...], tail], axis=0)

    paths_time_first = run(x0, eps)
    block_until_ready_tree(paths_time_first)
    paths = jnp.transpose(paths_time_first, (1, 0, 2))
    return np.asarray(times), np.asarray(paths)


def make_increment_table(times, paths):
    """Preserve trajectory/time identities for leakage-safe SINDy CV."""
    paths = np.asarray(paths, dtype=np.float32)
    times = np.asarray(times, dtype=np.float32)
    n_traj, n_times, d = paths.shape
    n_steps = n_times - 1

    t0 = np.broadcast_to(times[:-1][None, :], (n_traj, n_steps)).reshape(-1)
    dt = np.broadcast_to(np.diff(times)[None, :], (n_traj, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, d)
    x1 = paths[:, 1:, :].reshape(-1, d)
    dx = x1 - x0
    trajectory_id = np.repeat(np.arange(n_traj, dtype=np.int32), n_steps)
    time_index = np.tile(np.arange(n_steps, dtype=np.int32), n_traj)

    return {
        "t0": t0.astype(np.float32),
        "x0": x0.astype(np.float32),
        "dt": dt.astype(np.float32),
        "dx": dx.astype(np.float32),
        "x1": x1.astype(np.float32),
        "trajectory_id": trajectory_id,
        "time_index": time_index,
    }


def table_length(table):
    return int(table["t0"].shape[0])


def subset_table(table, indices):
    indices = np.asarray(indices, dtype=np.int64)
    return {key: np.asarray(value)[indices] for key, value in table.items()}


def split_increment_table(table, train_fraction, seed):
    """Manuscript-compatible random 80/20 split over one-step increments."""
    n = table_length(table)
    rng = np.random.default_rng(int(seed))
    permutation = rng.permutation(n)
    n_train = int(math.floor(float(train_fraction) * n))
    train_idx = permutation[:n_train]
    test_idx = permutation[n_train:]
    return subset_table(table, train_idx), subset_table(table, test_idx)


def prepare_case_data(benchmark_name, regime, seed):
    spec = BENCHMARKS[benchmark_name]
    d = int(spec["d"])
    n_pool = regime_trajectory_count(d, "efficient")
    n_used = regime_trajectory_count(d, regime)

    pool_seed = stable_seed("trajectory_pool", benchmark_name, seed)
    times, full_paths = simulate_sde_paths(spec, n_pool, pool_seed)

    # Nested design: scarce trajectories are the first subset of the same pool.
    paths = full_paths[:n_used]
    full_table = make_increment_table(times, paths)
    split_seed = stable_seed("increment_split", benchmark_name, regime, seed)
    train_table, test_table = split_increment_table(
        full_table,
        CONFIG["TRAIN_FRACTION"],
        split_seed,
    )

    return {
        "times": times,
        "paths": paths,
        "full_table": full_table,
        "train_table": train_table,
        "test_table": test_table,
        "n_trajectories": int(n_used),
        "n_pool_trajectories": int(n_pool),
    }


def table_to_jax_batch(table):
    return (
        jnp.asarray(table["t0"], dtype=jnp.float32),
        jnp.asarray(table["x0"], dtype=jnp.float32),
        jnp.asarray(table["dt"], dtype=jnp.float32),
        jnp.asarray(table["dx"], dtype=jnp.float32),
    )


def sample_jax_batch(key, batch, batch_size):
    n = int(batch[0].shape[0])
    idx = jax.random.randint(key, (int(batch_size),), 0, n)
    return tuple(array[idx] for array in batch)


def sample_tx_box_jax(key, n_points, spec):
    n_points = int(n_points)
    d = int(spec["d"])
    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t,
        (n_points,),
        minval=0.0,
        maxval=float(spec["t_final"]),
        dtype=jnp.float32,
    )
    x_lo, x_hi = map(float, spec["x_box"])
    x = jax.random.uniform(
        key_x,
        (n_points, d),
        minval=x_lo,
        maxval=x_hi,
        dtype=jnp.float32,
    )
    return jnp.concatenate([t[:, None], x], axis=1)


def sample_tx_jitter_jax(key, train_batch, n_points, jitter_std):
    key_idx, key_noise = jax.random.split(key)
    n = int(train_batch[0].shape[0])
    idx = jax.random.randint(key_idx, (int(n_points),), 0, n)
    t = train_batch[0][idx]
    x = train_batch[1][idx]
    noise = float(jitter_std) * jax.random.normal(key_noise, x.shape, dtype=x.dtype)
    return jnp.concatenate([t[:, None], x + noise], axis=1)



## Cell 4 — Dimension-generic neural SDE, Euler--Maruyama NLL, Stage-I training

In [ ]:
# =============================================================================
# Cell 4 — Dimension-generic neural SDE, Euler--Maruyama NLL, Stage-I training
# =============================================================================


def glorot_uniform(key, in_dim, out_dim):
    limit = jnp.sqrt(6.0 / float(in_dim + out_dim))
    return jax.random.uniform(
        key,
        (in_dim, out_dim),
        minval=-limit,
        maxval=limit,
        dtype=jnp.float32,
    )


class MLP:
    def __init__(self, in_dim, hidden_dim, depth, out_dim):
        self.in_dim = int(in_dim)
        self.hidden_dim = int(hidden_dim)
        self.depth = int(depth)
        self.out_dim = int(out_dim)

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)
        params = []
        for layer_key, in_dim, out_dim in zip(keys, sizes[:-1], sizes[1:]):
            params.append(
                {
                    "W": glorot_uniform(layer_key, in_dim, out_dim),
                    "b": jnp.zeros((out_dim,), dtype=jnp.float32),
                }
            )
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        return h @ params[-1]["W"] + params[-1]["b"]


def make_surrogate_model(spec):
    d = int(spec["d"])
    return MLP(
        in_dim=d + 1,
        hidden_dim=int(spec["hidden_dim"]),
        depth=int(spec["depth"]),
        out_dim=2 * d,
    )


def normalize_surrogate_input(spec, t, x):
    tn = jnp.asarray(t / float(spec["t_final"]), dtype=jnp.float32)
    xn = jnp.asarray(x / float(spec["input_x_scale"]), dtype=jnp.float32)
    return jnp.concatenate([tn[None], xn], axis=0)


def surrogate_f_sigma_one(model, params, spec, t, x):
    d = int(spec["d"])
    z = normalize_surrogate_input(spec, t, x)
    raw = model.apply(params, z)
    f = raw[:d]
    sigma = jax.nn.softplus(raw[d:]) + float(CONFIG["SIGMA_EPS"])
    return f, sigma


def surrogate_f_sigma_batch(model, params, spec, t, x):
    return jax.vmap(lambda tt, xx: surrogate_f_sigma_one(model, params, spec, tt, xx))(t, x)


def data_nll_loss(model, params, spec, batch):
    t0, x0, dt, dx = batch
    f, sigma = surrogate_f_sigma_batch(model, params, spec, t0, x0)
    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1.0e-8
    nll = 0.5 * ((dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var))
    return jnp.mean(nll)


def adam_init(params):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, params),
        "v": jax.tree_util.tree_map(jnp.zeros_like, params),
        "t": jnp.asarray(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1.0e-8):
    t_new = state["t"] + 1
    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"],
        grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * g * g,
        state["v"],
        grads,
    )
    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)
    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - float(lr) * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )
    return params_new, {"m": m_new, "v": v_new, "t": t_new}


def train_vanilla_neural_sde(spec, train_table, test_table, seed):
    model = make_surrogate_model(spec)
    train_batch = table_to_jax_batch(train_table)
    test_batch = table_to_jax_batch(test_table)

    key = jax.random.PRNGKey(stable_seed("vanilla_init", seed, spec["display_name"]))
    key, init_key = jax.random.split(key)
    params = model.init(init_key)
    opt_state = adam_init(params)

    epochs = effective_count(
        "baseline_epochs",
        spec["baseline_epochs"],
        CONFIG["SMOKE_BASELINE_EPOCHS"],
    )
    batch_size = min(int(CONFIG["BASELINE_BATCH_SIZE"]), table_length(train_table))
    print_every = max(1, min(int(CONFIG["BASELINE_PRINT_EVERY"]), epochs))

    @jax.jit
    def train_step(params, opt_state, batch):
        loss, grads = jax.value_and_grad(
            lambda p: data_nll_loss(model, p, spec, batch)
        )(params)
        params, opt_state = adam_update(
            params,
            grads,
            opt_state,
            CONFIG["BASELINE_LR"],
        )
        return params, opt_state, loss

    start = time.perf_counter()
    history = []

    for epoch in range(1, epochs + 1):
        key, batch_key = jax.random.split(key)
        batch = sample_jax_batch(batch_key, train_batch, batch_size)
        params, opt_state, loss = train_step(params, opt_state, batch)
        history.append(float(loss))

        if epoch == 1 or epoch % print_every == 0 or epoch == epochs:
            train_nll = data_nll_loss(model, params, spec, train_batch)
            test_nll = data_nll_loss(model, params, spec, test_batch)
            block_until_ready_tree((train_nll, test_nll))
            print(
                f"[Stage I | {spec['display_name']}] epoch {epoch:5d}/{epochs} | "
                f"batch={float(loss):.6e} | train={float(train_nll):.6e} | "
                f"test={float(test_nll):.6e}"
            )

    block_until_ready_tree(params)
    elapsed = time.perf_counter() - start

    return {
        "model": model,
        "params": params,
        "train_batch": train_batch,
        "test_batch": test_batch,
        "history": history,
        "seconds": float(elapsed),
        "train_nll": float(data_nll_loss(model, params, spec, train_batch)),
        "test_nll": float(data_nll_loss(model, params, spec, test_batch)),
    }


## Cell 5 — Dimension-generic editable projectable generator basis

In [ ]:
# =============================================================================
# Cell 5 — Dimension-generic editable projectable generator basis
# =============================================================================


def safe_exp(z):
    clip = float(CONFIG["BASIS_EXP_CLIP"])
    return jnp.exp(jnp.clip(z, -clip, clip))


def unit_vector(d, i, dtype=jnp.float32):
    return jnp.zeros((d,), dtype=dtype).at[int(i)].set(1.0)


def build_symmetry_basis(spec):
    """
    Build the same low-dimensional library families used by the submitted
    implementation and a structured 20D extension with R=208.

    Every basis vector returns z-components [tau, xi_1, ..., xi_d]. All xi are
    affine/linear in x, so the Ito Hessian term of xi is exactly zero.
    """
    d = int(spec["d"])
    mu_list = [float(mu) for mu in spec["basis_mu_list"]]
    basis = []

    def add(name, family, fn):
        basis.append({"name": str(name), "family": str(family), "fn": fn})

    # ---------------------------------------------------------------------
    # Core projectable generators
    # ---------------------------------------------------------------------
    add(
        "time_translation__dt",
        "core",
        lambda t, x: jnp.concatenate(
            [jnp.asarray([1.0], dtype=x.dtype), jnp.zeros((d,), dtype=x.dtype)]
        ),
    )

    for i in range(d):
        add(
            f"space_translation__dx{i+1}",
            "core",
            lambda t, x, i=i: jnp.concatenate(
                [jnp.asarray([0.0], dtype=x.dtype), unit_vector(d, i, x.dtype)]
            ),
        )

    add(
        "brownian_scaling__2t_dt_plus_x_dx",
        "core",
        lambda t, x: jnp.concatenate(
            [jnp.asarray([2.0 * t], dtype=x.dtype), x]
        ),
    )

    # ---------------------------------------------------------------------
    # Exact 1D library used in the manuscript summary: R=15.
    # ---------------------------------------------------------------------
    if d == 1:
        add(
            "gaeta_mu_plus1_translation__exp_t_dx",
            "canonical_linear",
            lambda t, x: jnp.asarray([0.0, safe_exp(t)], dtype=x.dtype),
        )
        add(
            "gaeta_mu_plus1_projective__exp_2t_dt_plus_xdx",
            "canonical_linear",
            lambda t, x: jnp.asarray(
                [safe_exp(2.0 * t), safe_exp(2.0 * t) * x[0]],
                dtype=x.dtype,
            ),
        )
        add(
            "gaeta_mu_minus1_translation__exp_minus_t_dx",
            "canonical_linear",
            lambda t, x: jnp.asarray([0.0, safe_exp(-t)], dtype=x.dtype),
        )
        add(
            "gaeta_mu_minus1_projective__exp_minus_2t_dt_minus_xdx",
            "canonical_linear",
            lambda t, x: jnp.asarray(
                [safe_exp(-2.0 * t), -safe_exp(-2.0 * t) * x[0]],
                dtype=x.dtype,
            ),
        )

        for mu in mu_list:
            add(
                f"generic_mu_{mu:+.2f}__exp_mut_dx",
                "generic_exponential",
                lambda t, x, mu=mu: jnp.asarray(
                    [0.0, safe_exp(mu * t)],
                    dtype=x.dtype,
                ),
            )
            add(
                f"generic_mu_{mu:+.2f}__exp_2mut_projective",
                "generic_exponential",
                lambda t, x, mu=mu: jnp.asarray(
                    [
                        safe_exp(2.0 * mu * t),
                        safe_exp(2.0 * mu * t) * mu * x[0],
                    ],
                    dtype=x.dtype,
                ),
            )

        if len(basis) != 15:
            raise RuntimeError(f"Expected 1D basis size 15, got {len(basis)}.")
        return basis

    # ---------------------------------------------------------------------
    # Common d>=3 extensions
    # ---------------------------------------------------------------------
    add(
        "time_scaling__t_dt",
        "time_distractor",
        lambda t, x: jnp.concatenate(
            [jnp.asarray([t], dtype=x.dtype), jnp.zeros((d,), dtype=x.dtype)]
        ),
    )
    add(
        "time_quadratic__t2_dt",
        "time_distractor",
        lambda t, x: jnp.concatenate(
            [jnp.asarray([t**2], dtype=x.dtype), jnp.zeros((d,), dtype=x.dtype)]
        ),
    )

    for i in range(d):
        add(
            f"component_scaling__x{i+1}_dx{i+1}",
            "component_scaling",
            lambda t, x, i=i: jnp.concatenate(
                [
                    jnp.asarray([0.0], dtype=x.dtype),
                    jnp.zeros((d,), dtype=x.dtype).at[i].set(x[i]),
                ]
            ),
        )

    for mu in mu_list:
        for i in range(d):
            add(
                f"mu_{mu:+.2f}__exp_mut_dx{i+1}",
                "generic_exponential_translation",
                lambda t, x, mu=mu, i=i: jnp.concatenate(
                    [
                        jnp.asarray([0.0], dtype=x.dtype),
                        jnp.zeros((d,), dtype=x.dtype).at[i].set(safe_exp(mu * t)),
                    ]
                ),
            )
        add(
            f"mu_{mu:+.2f}__exp_2mut_projective",
            "generic_exponential_projective",
            lambda t, x, mu=mu: jnp.concatenate(
                [
                    jnp.asarray([safe_exp(2.0 * mu * t)], dtype=x.dtype),
                    safe_exp(2.0 * mu * t) * mu * x,
                ]
            ),
        )

    # ---------------------------------------------------------------------
    # 3D: difference translations, all coordinate rotations, and directed
    # pair shears. The total is R=38.
    # ---------------------------------------------------------------------
    if d == 3:
        for i in range(d):
            j = (i + 1) % d
            add(
                f"cyclic_difference_translation__dx{i+1}_minus_dx{j+1}",
                "difference_translation",
                lambda t, x, i=i, j=j: jnp.concatenate(
                    [
                        jnp.asarray([0.0], dtype=x.dtype),
                        unit_vector(d, i, x.dtype) - unit_vector(d, j, x.dtype),
                    ]
                ),
            )

        for i in range(d):
            for j in range(i + 1, d):
                add(
                    f"rotation__x{j+1}_dx{i+1}_minus_x{i+1}_dx{j+1}",
                    "rotation",
                    lambda t, x, i=i, j=j: jnp.concatenate(
                        [
                            jnp.asarray([0.0], dtype=x.dtype),
                            jnp.zeros((d,), dtype=x.dtype)
                            .at[i].set(x[j])
                            .at[j].set(-x[i]),
                        ]
                    ),
                )
                add(
                    f"shear__x{j+1}_dx{i+1}",
                    "shear",
                    lambda t, x, i=i, j=j: jnp.concatenate(
                        [
                            jnp.asarray([0.0], dtype=x.dtype),
                            jnp.zeros((d,), dtype=x.dtype).at[i].set(x[j]),
                        ]
                    ),
                )
                add(
                    f"shear__x{i+1}_dx{j+1}",
                    "shear",
                    lambda t, x, i=i, j=j: jnp.concatenate(
                        [
                            jnp.asarray([0.0], dtype=x.dtype),
                            jnp.zeros((d,), dtype=x.dtype).at[j].set(x[i]),
                        ]
                    ),
                )

        if len(basis) != 38:
            raise RuntimeError(f"Expected 3D basis size 38, got {len(basis)}.")
        return basis

    # ---------------------------------------------------------------------
    # 20D: four generic ring-local linear banks, giving R=208.
    # These directions are candidate generator directions, not coefficient
    # features used by SINDy.
    # ---------------------------------------------------------------------
    if d == 20:
        for i in range(d):
            j = (i + 1) % d
            add(
                f"ring_difference_translation__dx{i+1}_minus_dx{j+1}",
                "ring_difference_translation",
                lambda t, x, i=i, j=j: jnp.concatenate(
                    [
                        jnp.asarray([0.0], dtype=x.dtype),
                        unit_vector(d, i, x.dtype) - unit_vector(d, j, x.dtype),
                    ]
                ),
            )

        for i in range(d):
            j = (i + 1) % d
            add(
                f"ring_rotation__x{j+1}_dx{i+1}_minus_x{i+1}_dx{j+1}",
                "ring_rotation",
                lambda t, x, i=i, j=j: jnp.concatenate(
                    [
                        jnp.asarray([0.0], dtype=x.dtype),
                        jnp.zeros((d,), dtype=x.dtype)
                        .at[i].set(x[j])
                        .at[j].set(-x[i]),
                    ]
                ),
            )

        for i in range(d):
            j = (i + 1) % d
            add(
                f"ring_forward_shear__x{j+1}_dx{i+1}",
                "ring_shear",
                lambda t, x, i=i, j=j: jnp.concatenate(
                    [
                        jnp.asarray([0.0], dtype=x.dtype),
                        jnp.zeros((d,), dtype=x.dtype).at[i].set(x[j]),
                    ]
                ),
            )

        for i in range(d):
            j = (i - 1) % d
            add(
                f"ring_backward_shear__x{j+1}_dx{i+1}",
                "ring_shear",
                lambda t, x, i=i, j=j: jnp.concatenate(
                    [
                        jnp.asarray([0.0], dtype=x.dtype),
                        jnp.zeros((d,), dtype=x.dtype).at[i].set(x[j]),
                    ]
                ),
            )

        if len(basis) != 208:
            raise RuntimeError(f"Expected 20D basis size 208, got {len(basis)}.")
        return basis

    # ---------------------------------------------------------------------
    # Optional 50D stress test: submitted R=333 construction.
    # ---------------------------------------------------------------------
    if d == 50:
        k = 6
        idx = jnp.arange(d, dtype=jnp.float32)
        directions = []
        for mode in range(1, k // 2 + 2):
            directions.append(jnp.sin(2.0 * jnp.pi * mode * idx / float(d)))
            if len(directions) < k:
                directions.append(jnp.cos(2.0 * jnp.pi * mode * idx / float(d)))
            if len(directions) >= k:
                break
        directions = [v / (jnp.linalg.norm(v) + 1.0e-8) for v in directions[:k]]

        for a in range(k):
            for b in range(a + 1, k):
                va, vb = directions[a], directions[b]
                add(
                    f"lowrank_rotation__v{a+1}_v{b+1}",
                    "lowrank_rotation",
                    lambda t, x, va=va, vb=vb: jnp.concatenate(
                        [
                            jnp.asarray([0.0], dtype=x.dtype),
                            jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                        ]
                    ),
                )

        for a in range(k - 1):
            va, vb = directions[a], directions[a + 1]
            add(
                f"lowrank_shear__v{a+2}_to_v{a+1}",
                "lowrank_shear",
                lambda t, x, va=va, vb=vb: jnp.concatenate(
                    [jnp.asarray([0.0], dtype=x.dtype), jnp.dot(vb, x) * va]
                ),
            )
            add(
                f"lowrank_shear__v{a+1}_to_v{a+2}",
                "lowrank_shear",
                lambda t, x, va=va, vb=vb: jnp.concatenate(
                    [jnp.asarray([0.0], dtype=x.dtype), jnp.dot(va, x) * vb]
                ),
            )

        if len(basis) != 333:
            raise RuntimeError(f"Expected 50D basis size 333, got {len(basis)}.")
        return basis

    raise ValueError(f"No symmetry basis specification for d={d}.")


def evaluate_basis_matrix(basis, t, x):
    return jnp.stack([entry["fn"](t, x) for entry in basis], axis=0)


## Cell 6 — Stage II discovery and Stage III Gaeta--Lie refinement

In [ ]:
# =============================================================================
# Cell 6 — Stage II discovery and Stage III Gaeta--Lie refinement
# =============================================================================


def generator_values_one(W, basis, z):
    t, x = z[0], z[1:]
    B = evaluate_basis_matrix(basis, t, x)
    return W @ B


def surrogate_derivatives_one(model, params, spec, z):
    def f_of_z(zz):
        return surrogate_f_sigma_one(model, params, spec, zz[0], zz[1:])[0]

    def sigma_of_z(zz):
        return surrogate_f_sigma_one(model, params, spec, zz[0], zz[1:])[1]

    f = f_of_z(z)
    sigma = sigma_of_z(z)
    Jf = jax.jacfwd(f_of_z)(z)
    Jsigma = jax.jacfwd(sigma_of_z)(z)
    return f, sigma, Jf, Jsigma


def determining_residuals_one(model, params, spec, W, basis, z):
    """Ito determining residual for diagonal Sigma and linear-in-x xi basis."""
    d = int(spec["d"])

    def generators_of_z(zz):
        return generator_values_one(W, basis, zz)

    generators = generators_of_z(z)            # (m, d+1)
    Jgenerators = jax.jacfwd(generators_of_z)(z)  # (m, d+1, d+1)

    tau = generators[:, 0]                     # (m,)
    xi = generators[:, 1:]                     # (m,d)
    tau_t = Jgenerators[:, 0, 0]              # (m,)
    tau_x = Jgenerators[:, 0, 1:]             # (m,d), exactly zero here
    xi_t = Jgenerators[:, 1:, 0]              # (m,d)
    xi_x = Jgenerators[:, 1:, 1:]             # (m,d,d)

    f, sigma, Jf, Jsigma = surrogate_derivatives_one(model, params, spec, z)
    f_t, f_x = Jf[:, 0], Jf[:, 1:]
    sigma_t, sigma_x = Jsigma[:, 0], Jsigma[:, 1:]

    # R_f^i = xi_t^i + f^j xi^i_{x_j} - xi^j f^i_{x_j}
    #         - tau f_t^i - f^i tau_t.
    # The Ito Hessian term is zero because every xi is affine in x.
    advect_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    Rf = (
        xi_t
        + advect_xi
        - xi_grad_f
        - tau[:, None] * f_t[None, :]
        - tau_t[:, None] * f[None, :]
    )

    # General diagonal-matrix residual R_Sigma^{i alpha}.
    first = jnp.einsum("p,aip->aip", sigma, xi_x)
    xi_grad_sigma = jnp.einsum("aj,ij->ai", xi, sigma_x)
    diagonal_scalar = (
        xi_grad_sigma
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )
    identity = jnp.eye(d, dtype=z.dtype)
    Rsigma = first - diagonal_scalar[:, :, None] * identity[None, :, :]

    projectable_residual = tau_x
    return Rf, Rsigma, projectable_residual


def determining_loss(model, params, spec, W, basis, tx):
    Rf, Rsigma, projectable = jax.vmap(
        lambda z: determining_residuals_one(model, params, spec, W, basis, z)
    )(tx)
    drift_term = jnp.mean(Rf**2)
    diffusion_term = jnp.mean(Rsigma**2)
    projectable_term = jnp.mean(projectable**2)
    return drift_term + diffusion_term, projectable_term, drift_term, diffusion_term


def generator_gram_loss(W, basis, tx):
    values = jax.vmap(lambda z: generator_values_one(W, basis, z))(tx)
    # values: (S,m,d+1)
    gram = jnp.einsum("saq,sbq->ab", values, values)
    gram = gram / float(values.shape[0] * values.shape[2])
    identity = jnp.eye(W.shape[0], dtype=W.dtype)
    independence = jnp.mean((gram - identity) ** 2)
    diagonal = jnp.diag(gram)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, float(CONFIG["MIN_GENERATOR_NORM"]) - diagonal) ** 2
    )
    return independence, norm_hinge, gram


def lie_bracket_one(W, basis, za, zb_unused=None):
    del zb_unused

    def generator_field(z):
        return generator_values_one(W, basis, z)

    X = generator_field(za)
    JX = jax.jacfwd(generator_field)(za)
    # bracket[a,b,q] = X_a^r d_r X_b^q - X_b^r d_r X_a^q
    return (
        jnp.einsum("ar,bqr->abq", X, JX)
        - jnp.einsum("br,aqr->abq", X, JX)
    )


def closure_loss(W, basis, tx):
    m = int(W.shape[0])
    if m < 2:
        return jnp.asarray(0.0, dtype=W.dtype)

    values = jax.vmap(lambda z: generator_values_one(W, basis, z))(tx)
    brackets = jax.vmap(lambda z: lie_bracket_one(W, basis, z))(tx)

    # Global least-squares projection of every bracket onto span{X_c}.
    A = jnp.transpose(values, (0, 2, 1)).reshape(-1, m)
    B = jnp.transpose(brackets, (0, 3, 1, 2)).reshape(-1, m * m)
    C = jnp.linalg.pinv(A, rtol=float(CONFIG["CLOSURE_RCOND"])) @ B
    residual = A @ C - B
    return jnp.mean(residual**2)


def discovery_objective(model, params, spec, W, basis, tx):
    det, projectable, drift_det, diffusion_det = determining_loss(
        model, params, spec, W, basis, tx
    )
    close = closure_loss(W, basis, tx)
    independence, norm_hinge, gram = generator_gram_loss(W, basis, tx)
    total = (
        float(CONFIG["LAMBDA_DET"]) * det
        + float(CONFIG["LAMBDA_CLOSE"]) * close
        + float(CONFIG["LAMBDA_IND"]) * independence
        + float(CONFIG["LAMBDA_NORM_HINGE"]) * norm_hinge
    )
    aux = {
        "det": det,
        "drift_det": drift_det,
        "diffusion_det": diffusion_det,
        "projectable": projectable,
        "close": close,
        "independence": independence,
        "norm_hinge": norm_hinge,
        "gram": gram,
    }
    return total, aux


def sample_symmetry_collocation(key, spec, train_batch, n_points):
    if spec.get("sym_collocation", "uniform_box") == "uniform_box":
        return sample_tx_box_jax(key, n_points, spec)
    if spec.get("sym_collocation") == "trajectory_jitter":
        return sample_tx_jitter_jax(
            key,
            train_batch,
            n_points,
            spec.get("sym_jitter_std", 0.05),
        )
    raise ValueError(f"Unknown symmetry collocation mode: {spec.get('sym_collocation')}")


def discover_symmetries(model, baseline_params, spec, basis, train_batch, seed):
    m = int(spec["n_generators"])
    R = len(basis)
    epochs = effective_count(
        "sym_epochs",
        spec["sym_epochs"],
        CONFIG["SMOKE_SYM_EPOCHS"],
    )
    n_points = int(spec["sym_points"])

    key = jax.random.PRNGKey(stable_seed("stage_ii", seed, spec["display_name"]))
    key, init_key = jax.random.split(key)
    W = float(CONFIG["SYM_INIT_SCALE"]) * jax.random.normal(
        init_key,
        (m, R),
        dtype=jnp.float32,
    )
    opt_state = adam_init(W)

    @jax.jit
    def step(W, opt_state, tx):
        (loss, aux), grads = jax.value_and_grad(
            lambda weights: discovery_objective(
                model,
                baseline_params,
                spec,
                weights,
                basis,
                tx,
            ),
            has_aux=True,
        )(W)
        W, opt_state = adam_update(
            W,
            grads,
            opt_state,
            CONFIG["SYM_DISCOVERY_LR"],
        )
        return W, opt_state, loss, aux

    print(f"Symmetry basis size R={R}; learned generators m={m}")
    print_every = max(1, min(250, epochs))
    history = []
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        key, collocation_key = jax.random.split(key)
        tx = sample_symmetry_collocation(
            collocation_key,
            spec,
            train_batch,
            n_points,
        )
        W, opt_state, loss, aux = step(W, opt_state, tx)
        history.append(float(loss))

        if epoch == 1 or epoch % print_every == 0 or epoch == epochs:
            block_until_ready_tree((W, loss, aux))
            print(
                f"[Stage II | {spec['display_name']}] epoch {epoch:5d}/{epochs} | "
                f"total={float(loss):.6e} | det={float(aux['det']):.6e} | "
                f"close={float(aux['close']):.6e} | ind={float(aux['independence']):.6e}"
            )

    block_until_ready_tree(W)
    elapsed = time.perf_counter() - start
    return {
        "W": W,
        "basis": basis,
        "history": history,
        "seconds": float(elapsed),
    }


def refine_gaeta_lie(model, baseline_params, spec, W, basis, train_batch, test_batch, seed):
    epochs = effective_count(
        "stage_f_epochs",
        spec["stage_f_epochs"],
        CONFIG["SMOKE_STAGE_F_EPOCHS"],
    )
    batch_size = min(int(CONFIG["STAGE_F_BATCH_SIZE"]), int(train_batch[0].shape[0]))
    n_extra = int(spec["stage_f_extra_points"])
    n_data_sym = min(n_extra, batch_size)

    params = jax.tree_util.tree_map(lambda value: value.copy(), baseline_params)
    opt_state = adam_init(params)
    key = jax.random.PRNGKey(stable_seed("stage_iii", seed, spec["display_name"]))

    @jax.jit
    def step(params, opt_state, data_batch, tx_sym):
        def objective(p):
            nll = data_nll_loss(model, p, spec, data_batch)
            det, projectable, drift_det, diffusion_det = determining_loss(
                model,
                p,
                spec,
                W,
                basis,
                tx_sym,
            )
            total = nll + float(CONFIG["STAGE_F_LAMBDA_SYM_DET"]) * det
            aux = {
                "nll": nll,
                "det": det,
                "drift_det": drift_det,
                "diffusion_det": diffusion_det,
                "projectable": projectable,
            }
            return total, aux

        (loss, aux), grads = jax.value_and_grad(objective, has_aux=True)(params)
        params, opt_state = adam_update(
            params,
            grads,
            opt_state,
            CONFIG["STAGE_F_LR"],
        )
        return params, opt_state, loss, aux

    print_every = max(1, min(250, epochs))
    history = []
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        key, batch_key, data_sym_key, extra_key = jax.random.split(key, 4)
        data_batch = sample_jax_batch(batch_key, train_batch, batch_size)

        data_sym_idx = jax.random.randint(
            data_sym_key,
            (n_data_sym,),
            0,
            data_batch[0].shape[0],
        )
        tx_data = jnp.concatenate(
            [
                data_batch[0][data_sym_idx, None],
                data_batch[1][data_sym_idx],
            ],
            axis=1,
        )
        tx_extra = sample_symmetry_collocation(
            extra_key,
            spec,
            train_batch,
            n_extra,
        )
        tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)

        params, opt_state, loss, aux = step(
            params,
            opt_state,
            data_batch,
            tx_sym,
        )
        history.append(float(loss))

        if epoch == 1 or epoch % print_every == 0 or epoch == epochs:
            test_nll = data_nll_loss(model, params, spec, test_batch)
            block_until_ready_tree((params, loss, aux, test_nll))
            print(
                f"[Stage III | {spec['display_name']}] epoch {epoch:5d}/{epochs} | "
                f"total={float(loss):.6e} | data={float(aux['nll']):.6e} | "
                f"det={float(aux['det']):.6e} | test={float(test_nll):.6e}"
            )

    block_until_ready_tree(params)
    elapsed = time.perf_counter() - start
    return {
        "params": params,
        "history": history,
        "seconds": float(elapsed),
        "train_nll": float(data_nll_loss(model, params, spec, train_batch)),
        "test_nll": float(data_nll_loss(model, params, spec, test_batch)),
    }


## Cell 7 — Standard and oracle-enriched SINDy coefficient libraries

In [ ]:
# =============================================================================
# Cell 7 — Standard and oracle-enriched SINDy coefficient libraries
# =============================================================================


@dataclass
class FeatureLibrary:
    name: str
    feature_names: tuple
    feature_functions: tuple

    @property
    def n_features(self):
        return len(self.feature_names)

    def transform(self, t, x):
        t = np.asarray(t, dtype=np.float64).reshape(-1)
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            x = x[:, None]
        if x.shape[0] != t.shape[0]:
            raise ValueError(f"t has {t.shape[0]} rows but x has {x.shape[0]} rows.")

        columns = []
        for name, function in zip(self.feature_names, self.feature_functions):
            values = np.asarray(function(t, x), dtype=np.float64).reshape(-1)
            if values.shape[0] != t.shape[0]:
                raise ValueError(
                    f"Feature {name!r} returned shape {values.shape}; "
                    f"expected {(t.shape[0],)}."
                )
            if not np.isfinite(values).all():
                raise FloatingPointError(f"Feature {name!r} produced non-finite values.")
            columns.append(values)
        return np.column_stack(columns)


def sigmoid_np(value):
    value = np.asarray(value, dtype=np.float64)
    positive = value >= 0.0
    output = np.empty_like(value)
    output[positive] = 1.0 / (1.0 + np.exp(-value[positive]))
    exp_value = np.exp(value[~positive])
    output[~positive] = exp_value / (1.0 + exp_value)
    return output


def build_sindy_library(benchmark_name, library_kind):
    """
    Build a standard or oracle-enriched coefficient library.

    The oracle library adds the correct nonlinear families and exact arguments,
    but never supplies the true numerical coefficients. Because stochastic
    SINDy directly estimates q_i=sigma_i^2, the oracle diffusion additions also
    include pairwise products of the primitive sigma features needed to span q.
    """
    if library_kind not in {"standard", "oracle"}:
        raise ValueError(f"Unknown library kind: {library_kind}")

    spec = BENCHMARKS[benchmark_name]
    d = int(spec["d"])
    names = []
    functions = []
    seen_names = set()

    def add(name, function):
        name = str(name)
        if name in seen_names:
            return
        seen_names.add(name)
        names.append(name)
        functions.append(function)

    # ---------------------------------------------------------------------
    # Standard polynomial/Fourier library fixed before inspecting any one SDE
    # ---------------------------------------------------------------------
    add("1", lambda t, x: np.ones_like(t))
    add("t", lambda t, x: t)
    add("t^2", lambda t, x: t**2)

    for omega in CONFIG["SINDY_STANDARD_TIME_FREQUENCIES"]:
        omega = float(omega)
        add(f"sin({omega:g} t)", lambda t, x, omega=omega: np.sin(omega * t))
        add(f"cos({omega:g} t)", lambda t, x, omega=omega: np.cos(omega * t))

    if d <= 3:
        # All state monomials through total degree three.
        for degree in (1, 2, 3):
            for combo in combinations_with_replacement(range(d), degree):
                powers = np.bincount(combo, minlength=d)
                pieces = []
                for coordinate, power in enumerate(powers):
                    if power == 1:
                        pieces.append(f"x{coordinate+1}")
                    elif power > 1:
                        pieces.append(f"x{coordinate+1}^{power}")
                feature_name = " ".join(pieces)
                add(
                    feature_name,
                    lambda t, x, powers=powers.copy(): np.prod(
                        np.power(x, powers[None, :]),
                        axis=1,
                    ),
                )
    else:
        # High-dimensional standard library requested in E5:
        # all linear terms, all quadratic terms, and coordinatewise cubics.
        for i in range(d):
            add(f"x{i+1}", lambda t, x, i=i: x[:, i])
        for i in range(d):
            for j in range(i, d):
                add(
                    f"x{i+1} x{j+1}",
                    lambda t, x, i=i, j=j: x[:, i] * x[:, j],
                )
        for i in range(d):
            add(f"x{i+1}^3", lambda t, x, i=i: x[:, i] ** 3)

    for i in range(d):
        add(f"sin(x{i+1})", lambda t, x, i=i: np.sin(x[:, i]))
        add(f"cos(x{i+1})", lambda t, x, i=i: np.cos(x[:, i]))

    # Selected low-order time-state products.
    for omega in CONFIG["SINDY_TIME_STATE_PRODUCT_FREQUENCIES"]:
        omega = float(omega)
        for i in range(d):
            add(
                f"x{i+1} sin({omega:g} t)",
                lambda t, x, omega=omega, i=i: x[:, i] * np.sin(omega * t),
            )
            add(
                f"x{i+1} cos({omega:g} t)",
                lambda t, x, omega=omega, i=i: x[:, i] * np.cos(omega * t),
            )

    if library_kind == "standard":
        library = FeatureLibrary(
            name="standard polynomial/Fourier",
            feature_names=tuple(names),
            feature_functions=tuple(functions),
        )
    else:
        # -----------------------------------------------------------------
        # Oracle-enriched additions for each benchmark.
        # -----------------------------------------------------------------
        if benchmark_name == "ex_1_1":
            add("sin(1.7 t)", lambda t, x: np.sin(1.7 * t))
            add("tanh(x1)", lambda t, x: np.tanh(x[:, 0]))

            g1 = lambda t, x: np.sin(t) ** 2
            g2 = lambda t, x: sigmoid_np(x[:, 0])
            add("sin(t)^2", g1)
            add("sigmoid(x1)", g2)
            add("sin(t)^4", lambda t, x: g1(t, x) ** 2)
            add("sigmoid(x1)^2", lambda t, x: g2(t, x) ** 2)
            add(
                "sin(t)^2 sigmoid(x1)",
                lambda t, x: g1(t, x) * g2(t, x),
            )

        elif benchmark_name == "ex_1_3":
            add("sin(1.5 t)", lambda t, x: np.sin(1.5 * t))
            add("tanh(1.4 x1)", lambda t, x: np.tanh(1.4 * x[:, 0]))
            # x1*sin(t) is already present in the fixed standard library.

            g1 = lambda t, x: np.cos(1.2 * t) ** 2
            g2 = lambda t, x: np.tanh(0.9 * x[:, 0]) ** 2
            g3 = lambda t, x: sigmoid_np(-x[:, 0] + 0.3 * np.sin(t))
            primitive = [
                ("cos(1.2 t)^2", g1),
                ("tanh(0.9 x1)^2", g2),
                ("sigmoid(-x1 + 0.3 sin(t))", g3),
            ]
            for primitive_name, primitive_fn in primitive:
                add(primitive_name, primitive_fn)
            for i, (name_i, fn_i) in enumerate(primitive):
                for j in range(i, len(primitive)):
                    name_j, fn_j = primitive[j]
                    add(
                        f"({name_i})({name_j})",
                        lambda t, x, fn_i=fn_i, fn_j=fn_j: fn_i(t, x) * fn_j(t, x),
                    )

        elif benchmark_name == "ex_3_1":
            add("cos(1.3 t)", lambda t, x: np.cos(1.3 * t))
            for i in range(d):
                add(f"tanh(x{i+1})", lambda t, x, i=i: np.tanh(x[:, i]))
                add(f"sin(x{i+1})^2", lambda t, x, i=i: np.sin(x[:, i]) ** 2)
                add(f"sin(x{i+1})^4", lambda t, x, i=i: np.sin(x[:, i]) ** 4)

        elif benchmark_name == "ex_3_2":
            add("sin(2 pi t)", lambda t, x: np.sin(2.0 * np.pi * t))
            add("cos(2 pi t)", lambda t, x: np.cos(2.0 * np.pi * t))
            for i in range(d):
                add(f"sigmoid(x{i+1})", lambda t, x, i=i: sigmoid_np(x[:, i]))
                add(
                    f"sigmoid(x{i+1})^2",
                    lambda t, x, i=i: sigmoid_np(x[:, i]) ** 2,
                )

        elif benchmark_name == "ex_20_3":
            theta = 2.0 * np.pi * np.arange(d, dtype=np.float64) / float(d)
            for i in range(d):
                add(f"tanh(x{i+1})", lambda t, x, i=i: np.tanh(x[:, i]))
                add(f"sin(x{i+1})^2", lambda t, x, i=i: np.sin(x[:, i]) ** 2)
                add(f"sin(x{i+1})^4", lambda t, x, i=i: np.sin(x[:, i]) ** 4)
                # The fixed standard sin(t), cos(t) pair already spans
                # every phase-shifted forcing sin(t+theta_i), avoiding an
                # exactly collinear bank of twenty phase copies.

        elif benchmark_name == "ex_50_1":
            theta = 2.0 * np.pi * np.arange(d, dtype=np.float64) / float(d)
            add("sin(2 pi t)", lambda t, x: np.sin(2.0 * np.pi * t))
            add("cos(2 pi t)", lambda t, x: np.cos(2.0 * np.pi * t))
            for i in range(d):
                add(f"sigmoid(x{i+1})", lambda t, x, i=i: sigmoid_np(x[:, i]))
                add(
                    f"sigmoid(x{i+1})^2",
                    lambda t, x, i=i: sigmoid_np(x[:, i]) ** 2,
                )
                # sin(2*pi*t), cos(2*pi*t) span every phase-shifted forcing.
        else:
            raise ValueError(f"No oracle enrichment specified for {benchmark_name}.")

        library = FeatureLibrary(
            name="oracle-enriched optimistic upper bound",
            feature_names=tuple(names),
            feature_functions=tuple(functions),
        )

    # Smoke mode keeps the first features plus all oracle-only tail features up
    # to a small cap. This is only for syntax/runtime checking, never reporting.
    if CONFIG["EXECUTION_MODE"] == "smoke":
        cap = min(int(CONFIG["SMOKE_MAX_SINDY_FEATURES"]), library.n_features)
        library = FeatureLibrary(
            name=library.name + " [SMOKE-TRUNCATED]",
            feature_names=library.feature_names[:cap],
            feature_functions=library.feature_functions[:cap],
        )

    return library


for benchmark_name in CONFIG["E5_BENCHMARKS"]:
    standard_library = build_sindy_library(benchmark_name, "standard")
    oracle_library = build_sindy_library(benchmark_name, "oracle")
    print(
        f"{benchmark_name}: standard R={standard_library.n_features}, "
        f"oracle R={oracle_library.n_features}"
    )

## Cell 8 — Boninsegna-style CV-SSR and stochastic Kramers--Moyal regression

In [ ]:
# =============================================================================
# Cell 8 — Boninsegna-style CV-SSR and stochastic Kramers--Moyal regression
# =============================================================================


@dataclass
class SSRPath:
    coefficients_by_size: dict
    active_indices_by_size: dict
    removed_feature_order: tuple


@dataclass
class CVSSRTargetFit:
    target_name: str
    coefficients: np.ndarray
    selected_size: int
    selected_by: str
    cv_mean_by_size: np.ndarray
    cv_se_by_size: np.ndarray
    cv_ratio_by_size: np.ndarray
    active_indices: np.ndarray
    removed_feature_order: tuple


@dataclass
class StochasticSINDyModel:
    benchmark_name: str
    library_kind: str
    library: FeatureLibrary
    drift_coefficients: np.ndarray
    q_coefficients: np.ndarray
    drift_fits: tuple
    q_fits: tuple
    fit_seconds: float

    def predict_f_q(self, t, x):
        t = np.asarray(t, dtype=np.float64).reshape(-1)
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            x = x[:, None]
        theta = self.library.transform(t, x)
        f = theta @ self.drift_coefficients.T
        q = theta @ self.q_coefficients.T
        return f, q

    def predict_f_sigma(self, t, x):
        f, q = self.predict_f_q(t, x)
        sigma = np.sqrt(np.maximum(q, float(CONFIG["Q_FLOOR"])))
        return f, sigma


def normalized_design_matrix(X):
    X = np.asarray(X, dtype=np.float64)
    scales = np.sqrt(np.mean(X**2, axis=0))
    scales = np.where(
        scales < float(CONFIG["SINDY_FEATURE_SCALE_FLOOR"]),
        1.0,
        scales,
    )
    return X / scales[None, :], scales


def least_squares_coefficients(X, y, active_indices):
    active_indices = np.asarray(active_indices, dtype=np.int64)
    if active_indices.size == 0:
        raise ValueError("SSR cannot fit an empty active set.")

    X_active = X[:, active_indices]
    X_normalized, scales = normalized_design_matrix(X_active)
    coefficients_normalized, *_ = np.linalg.lstsq(
        X_normalized,
        y,
        rcond=float(CONFIG["SINDY_LSTSQ_RCOND"]),
    )
    coefficients_physical = coefficients_normalized / scales

    full_physical = np.zeros(X.shape[1], dtype=np.float64)
    full_normalized = np.zeros(X.shape[1], dtype=np.float64)
    full_physical[active_indices] = coefficients_physical
    full_normalized[active_indices] = coefficients_normalized
    return full_physical, full_normalized


def fit_ssr_path(X, y, constant_index=0):
    """
    Stepwise Sparse Regression path.

    Start with every feature, fit ordinary least squares, remove exactly the
    smallest-magnitude normalized active coefficient, refit, and continue until
    one term remains. Removed terms never re-enter.
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    if X.shape[0] != y.shape[0]:
        raise ValueError("X and y row counts differ.")

    n_features = X.shape[1]
    active = np.arange(n_features, dtype=np.int64)
    coefficients_by_size = {}
    active_by_size = {}
    removed = []

    while active.size >= 1:
        coefficients_physical, coefficients_normalized = least_squares_coefficients(
            X,
            y,
            active,
        )
        size = int(active.size)
        coefficients_by_size[size] = coefficients_physical
        active_by_size[size] = active.copy()

        if size == 1:
            break

        removable = active.copy()
        if bool(CONFIG["SINDY_PROTECT_CONSTANT"]) and constant_index in removable and removable.size > 1:
            removable = removable[removable != int(constant_index)]

        local_magnitudes = np.abs(coefficients_normalized[removable])
        remove_index = int(removable[int(np.argmin(local_magnitudes))])
        removed.append(remove_index)
        active = active[active != remove_index]

    return SSRPath(
        coefficients_by_size=coefficients_by_size,
        active_indices_by_size=active_by_size,
        removed_feature_order=tuple(removed),
    )


def build_leakage_safe_cv_folds(train_table, requested_folds, seed):
    """
    Use trajectory-grouped folds when possible. With a single trajectory,
    use contiguous temporal blocks rather than random transition-level folds.
    """
    trajectory_id = np.asarray(train_table["trajectory_id"], dtype=np.int64)
    time_index = np.asarray(train_table["time_index"], dtype=np.int64)
    unique_trajectories = np.unique(trajectory_id)
    rng = np.random.default_rng(int(seed))

    folds = []

    if unique_trajectories.size >= 2:
        shuffled = unique_trajectories.copy()
        rng.shuffle(shuffled)
        n_folds = min(int(requested_folds), int(shuffled.size))
        trajectory_groups = np.array_split(shuffled, n_folds)
        for validation_trajectories in trajectory_groups:
            validation_mask = np.isin(trajectory_id, validation_trajectories)
            validation_indices = np.flatnonzero(validation_mask)
            training_indices = np.flatnonzero(~validation_mask)
            if training_indices.size and validation_indices.size:
                folds.append((training_indices, validation_indices))
    else:
        order = np.argsort(time_index, kind="stable")
        n_folds = min(int(requested_folds), max(2, int(order.size // 20)))
        for validation_indices in np.array_split(order, n_folds):
            validation_indices = np.asarray(validation_indices, dtype=np.int64)
            training_mask = np.ones(order.size, dtype=bool)
            training_mask[validation_indices] = False
            training_indices = np.flatnonzero(training_mask)
            if training_indices.size and validation_indices.size:
                folds.append((training_indices, validation_indices))

    if len(folds) < 2:
        raise RuntimeError(
            "Could not construct at least two nonempty leakage-safe CV folds."
        )
    return folds


def select_cv_ssr_size(cv_mean, cv_se):
    """Ratio-elbow selection with a one-standard-error fallback."""
    cv_mean = np.asarray(cv_mean, dtype=np.float64)
    cv_se = np.asarray(cv_se, dtype=np.float64)
    n_features = cv_mean.size

    # Array index n-1 corresponds to n active terms.
    ratios = np.full(n_features, np.nan, dtype=np.float64)
    for n in range(2, n_features + 1):
        denominator = max(cv_mean[n - 1], 1.0e-30)
        ratios[n - 1] = cv_mean[n - 2] / denominator

    minimum_index = int(np.nanargmin(cv_mean))
    minimum_error = float(cv_mean[minimum_index])
    near_minimum_limit = minimum_error * (
        1.0 + float(CONFIG["SINDY_CV_NEAR_MIN_REL_TOL"])
    )

    elbow_candidates = []
    for n in range(2, n_features + 1):
        ratio = ratios[n - 1]
        if (
            np.isfinite(ratio)
            and ratio >= float(CONFIG["SINDY_CV_RATIO_THRESHOLD"])
            and cv_mean[n - 1] <= near_minimum_limit
        ):
            elbow_candidates.append(n)

    if elbow_candidates:
        selected_size = int(min(elbow_candidates))
        selected_by = "CV ratio elbow"
    elif bool(CONFIG["SINDY_USE_ONE_STANDARD_ERROR_FALLBACK"]):
        threshold = minimum_error + float(cv_se[minimum_index])
        eligible = np.flatnonzero(cv_mean <= threshold) + 1
        selected_size = int(np.min(eligible))
        selected_by = "one-standard-error fallback"
    else:
        selected_size = minimum_index + 1
        selected_by = "minimum CV error"

    return selected_size, selected_by, ratios


def fit_cv_ssr_target(X, y, folds, target_name):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    n_features = X.shape[1]
    fold_errors = np.full((len(folds), n_features), np.nan, dtype=np.float64)

    for fold_index, (training_indices, validation_indices) in enumerate(folds):
        X_train = X[training_indices]
        y_train = y[training_indices]
        X_validation = X[validation_indices]
        y_validation = y[validation_indices]

        path = fit_ssr_path(X_train, y_train)
        target_variance = max(float(np.var(y_validation)), 1.0e-12)

        for size in range(1, n_features + 1):
            coefficients = path.coefficients_by_size[size]
            prediction = X_validation @ coefficients
            mse = float(np.mean((prediction - y_validation) ** 2))
            # Normalization makes CV scales comparable across drift/diffusion
            # coordinates without changing the selected path within a target.
            fold_errors[fold_index, size - 1] = mse / target_variance

    cv_mean = np.mean(fold_errors, axis=0)
    if len(folds) > 1:
        cv_se = np.std(fold_errors, axis=0, ddof=1) / np.sqrt(len(folds))
    else:
        cv_se = np.zeros_like(cv_mean)

    selected_size, selected_by, ratios = select_cv_ssr_size(cv_mean, cv_se)
    full_path = fit_ssr_path(X, y)
    coefficients = full_path.coefficients_by_size[selected_size].copy()
    coefficients[
        np.abs(coefficients) < float(CONFIG["SINDY_COEF_ZERO_TOL"])
    ] = 0.0
    active_indices = np.flatnonzero(coefficients != 0.0)

    return CVSSRTargetFit(
        target_name=str(target_name),
        coefficients=coefficients,
        selected_size=int(selected_size),
        selected_by=str(selected_by),
        cv_mean_by_size=cv_mean,
        cv_se_by_size=cv_se,
        cv_ratio_by_size=ratios,
        active_indices=active_indices,
        removed_feature_order=full_path.removed_feature_order,
    )


def fit_stochastic_sindy(benchmark_name, library_kind, train_table, seed):
    """
    Fit drift f_i and diagonal covariance q_i=sigma_i^2 using

        Y_f_i = Delta X_i / Delta t,
        Y_q_i = (Delta X_i)^2 / Delta t,

    followed by leakage-safe cross-validated SSR for each target equation.
    """
    spec = BENCHMARKS[benchmark_name]
    d = int(spec["d"])
    library = build_sindy_library(benchmark_name, library_kind)

    start = time.perf_counter()
    X_library = library.transform(train_table["t0"], train_table["x0"])
    dt = np.asarray(train_table["dt"], dtype=np.float64)[:, None]
    dx = np.asarray(train_table["dx"], dtype=np.float64)
    drift_targets = dx / dt
    q_targets = dx**2 / dt

    folds = build_leakage_safe_cv_folds(
        train_table,
        CONFIG["SINDY_CV_FOLDS"],
        stable_seed("sindy_cv", benchmark_name, library_kind, seed),
    )

    print(
        f"[SINDy {library_kind} | {spec['display_name']}] "
        f"N={X_library.shape[0]}, R={X_library.shape[1]}, folds={len(folds)}"
    )

    drift_fits = []
    q_fits = []

    for component in range(d):
        fit = fit_cv_ssr_target(
            X_library,
            drift_targets[:, component],
            folds,
            target_name=f"f_{component+1}",
        )
        drift_fits.append(fit)
        print(
            f"  drift {component+1:02d}/{d}: n={fit.selected_size:3d} "
            f"({fit.selected_by})"
        )

    for component in range(d):
        fit = fit_cv_ssr_target(
            X_library,
            q_targets[:, component],
            folds,
            target_name=f"q_{component+1}",
        )
        q_fits.append(fit)
        print(
            f"  q     {component+1:02d}/{d}: n={fit.selected_size:3d} "
            f"({fit.selected_by})"
        )

    drift_coefficients = np.stack([fit.coefficients for fit in drift_fits], axis=0)
    q_coefficients = np.stack([fit.coefficients for fit in q_fits], axis=0)
    elapsed = time.perf_counter() - start

    return StochasticSINDyModel(
        benchmark_name=benchmark_name,
        library_kind=library_kind,
        library=library,
        drift_coefficients=drift_coefficients,
        q_coefficients=q_coefficients,
        drift_fits=tuple(drift_fits),
        q_fits=tuple(q_fits),
        fit_seconds=float(elapsed),
    )


def print_sindy_equations(model, top_k=None):
    top_k = int(top_k or CONFIG["PRINT_TOP_SINDY_TERMS"])
    names = np.asarray(model.library.feature_names, dtype=object)
    print(
        f"\nLearned equations: {model.benchmark_name} | "
        f"{model.library_kind} | {model.library.name}"
    )

    for target_type, coefficients in [
        ("drift", model.drift_coefficients),
        ("q=sigma^2", model.q_coefficients),
    ]:
        for component, row in enumerate(coefficients):
            active = np.flatnonzero(np.abs(row) >= float(CONFIG["SINDY_COEF_ZERO_TOL"]))
            order = active[np.argsort(-np.abs(row[active]))] if active.size else active
            pieces = [f"{row[index]:+.5e}*{names[index]}" for index in order[:top_k]]
            print(
                f"  {target_type}[{component+1}] | nnz={active.size}: "
                + (" ".join(pieces) if pieces else "0")
            )


def save_sindy_diagnostics(model, benchmark_name, regime, seed):
    output_prefix = (
        RESULTS_DIR
        / f"{benchmark_name}__{regime}__seed{seed}__sindy_{model.library_kind}"
    )

    coefficient_rows = []
    for target_type, coefficients, fits in [
        ("drift", model.drift_coefficients, model.drift_fits),
        ("q", model.q_coefficients, model.q_fits),
    ]:
        for component, (coefficient_row, fit) in enumerate(zip(coefficients, fits), start=1):
            for feature_index, (feature_name, coefficient) in enumerate(
                zip(model.library.feature_names, coefficient_row)
            ):
                coefficient_rows.append(
                    {
                        "benchmark": benchmark_name,
                        "regime": regime,
                        "seed": int(seed),
                        "library_kind": model.library_kind,
                        "target_type": target_type,
                        "component": component,
                        "feature_index": feature_index,
                        "feature_name": feature_name,
                        "coefficient": float(coefficient),
                        "selected_size": int(fit.selected_size),
                        "selected_by": fit.selected_by,
                        "active": bool(abs(coefficient) >= CONFIG["SINDY_COEF_ZERO_TOL"]),
                    }
                )

    coefficient_path = Path(str(output_prefix) + "__coefficients.csv")
    pd.DataFrame(coefficient_rows).to_csv(coefficient_path, index=False)

    cv_rows = []
    for target_type, fits in [("drift", model.drift_fits), ("q", model.q_fits)]:
        for component, fit in enumerate(fits, start=1):
            for size in range(1, len(fit.cv_mean_by_size) + 1):
                cv_rows.append(
                    {
                        "benchmark": benchmark_name,
                        "regime": regime,
                        "seed": int(seed),
                        "library_kind": model.library_kind,
                        "target_type": target_type,
                        "component": component,
                        "solution_size": size,
                        "cv_mean": float(fit.cv_mean_by_size[size - 1]),
                        "cv_se": float(fit.cv_se_by_size[size - 1]),
                        "cv_ratio_previous_over_current": (
                            float(fit.cv_ratio_by_size[size - 1])
                            if np.isfinite(fit.cv_ratio_by_size[size - 1])
                            else np.nan
                        ),
                        "selected": bool(size == fit.selected_size),
                        "selected_by": fit.selected_by,
                    }
                )

    cv_path = Path(str(output_prefix) + "__cv_paths.csv")
    pd.DataFrame(cv_rows).to_csv(cv_path, index=False)
    return str(coefficient_path), str(cv_path)

## Cell 9 — Shared coefficient, one-step, rollout, runtime, and sparsity evaluation

In [ ]:
# =============================================================================
# Cell 9 — Shared coefficient, one-step, rollout, runtime, and sparsity evaluation
# =============================================================================


def make_true_predictor(spec):
    drift_fn = spec["drift_fn"]
    sigma_fn = spec["sigma_fn"]

    @jax.jit
    def evaluate(t, x):
        return jax.vmap(lambda tt, xx: (drift_fn(tt, xx), sigma_fn(tt, xx)))(t, x)

    def predict_f_sigma(t, x):
        t_array = np.asarray(t, dtype=np.float32).reshape(-1)
        x_array = np.asarray(x, dtype=np.float32)
        if x_array.ndim == 1:
            x_array = x_array[:, None]
        f, sigma = evaluate(jnp.asarray(t_array), jnp.asarray(x_array))
        block_until_ready_tree((f, sigma))
        return np.asarray(f, dtype=np.float64), np.asarray(sigma, dtype=np.float64)

    def predict_f_q(t, x):
        f, sigma = predict_f_sigma(t, x)
        return f, sigma**2

    return {"predict_f_sigma": predict_f_sigma, "predict_f_q": predict_f_q}


def make_neural_predictor(model, params, spec):
    @jax.jit
    def evaluate(t, x):
        return surrogate_f_sigma_batch(model, params, spec, t, x)

    def predict_f_sigma(t, x):
        t_array = np.asarray(t, dtype=np.float32).reshape(-1)
        x_array = np.asarray(x, dtype=np.float32)
        if x_array.ndim == 1:
            x_array = x_array[:, None]
        f, sigma = evaluate(jnp.asarray(t_array), jnp.asarray(x_array))
        block_until_ready_tree((f, sigma))
        return np.asarray(f, dtype=np.float64), np.asarray(sigma, dtype=np.float64)

    def predict_f_q(t, x):
        f, sigma = predict_f_sigma(t, x)
        return f, sigma**2

    return {"predict_f_sigma": predict_f_sigma, "predict_f_q": predict_f_q}


def make_sindy_predictor(model):
    return {
        "predict_f_sigma": model.predict_f_sigma,
        "predict_f_q": model.predict_f_q,
    }


def make_evaluation_bundle(spec, benchmark_name, regime, seed, test_table):
    d = int(spec["d"])
    if CONFIG["EXECUTION_MODE"] == "smoke":
        n_box = int(CONFIG["SMOKE_N_EVAL_POINTS_BOX"])
        n_support = int(CONFIG["SMOKE_N_EVAL_POINTS_SUPPORT"])
        n_rollout = int(CONFIG["SMOKE_N_ROLLOUT_PATHS"])
    else:
        n_box = int(CONFIG["N_EVAL_POINTS_BOX"])
        n_support = int(CONFIG["N_EVAL_POINTS_SUPPORT"])
        n_rollout = int(CONFIG["N_ROLLOUT_PATHS"])

    rng_box = np.random.default_rng(
        stable_seed("evaluation_box", benchmark_name, regime, seed)
    )
    t_box = rng_box.uniform(0.0, float(spec["t_final"]), size=n_box)
    x_lo, x_hi = map(float, spec["x_box"])
    x_box = rng_box.uniform(x_lo, x_hi, size=(n_box, d))

    rng_support = np.random.default_rng(
        stable_seed("evaluation_support", benchmark_name, regime, seed)
    )
    support_indices = rng_support.choice(
        table_length(test_table),
        size=n_support,
        replace=table_length(test_table) < n_support,
    )
    t_support = np.asarray(test_table["t0"])[support_indices]
    x_support = np.asarray(test_table["x0"])[support_indices]

    rng_rollout = np.random.default_rng(
        stable_seed("evaluation_rollout", benchmark_name, regime, seed)
    )
    x0_mean = np.asarray(spec["x0_mean_array"], dtype=np.float64)
    x0_std = np.asarray(spec["x0_std_array"], dtype=np.float64)
    rollout_x0 = x0_mean + x0_std * rng_rollout.normal(size=(n_rollout, d))
    n_steps = int(round(float(spec["t_final"]) / float(spec["dt"])))
    rollout_noise = rng_rollout.normal(size=(n_steps, n_rollout, d))

    return {
        "t_box": t_box,
        "x_box": x_box,
        "t_support": t_support,
        "x_support": x_support,
        "rollout_x0": rollout_x0,
        "rollout_noise": rollout_noise,
    }


def coefficient_metrics(predictor, truth_predictor, t, x):
    f_true, sigma_true = truth_predictor["predict_f_sigma"](t, x)
    f_pred, sigma_pred = predictor["predict_f_sigma"](t, x)
    _, q_pred_raw = predictor["predict_f_q"](t, x)
    q_true = sigma_true**2

    return {
        "drift_mse": float(np.mean((f_pred - f_true) ** 2)),
        "sigma_mse": float(np.mean((sigma_pred - sigma_true) ** 2)),
        "q_mse": float(np.mean((q_pred_raw - q_true) ** 2)),
        "negative_q_fraction": float(np.mean(q_pred_raw < 0.0)),
        "q_clip_fraction": float(np.mean(q_pred_raw < float(CONFIG["Q_FLOOR"]))),
    }


def one_step_metrics(predictor, test_table):
    t = np.asarray(test_table["t0"], dtype=np.float64)
    x = np.asarray(test_table["x0"], dtype=np.float64)
    dt = np.asarray(test_table["dt"], dtype=np.float64)[:, None]
    dx = np.asarray(test_table["dx"], dtype=np.float64)

    f, sigma = predictor["predict_f_sigma"](t, x)
    mean = f * dt
    variance = np.maximum(sigma**2 * dt, 1.0e-12)
    nll = 0.5 * (
        (dx - mean) ** 2 / variance
        + np.log(2.0 * np.pi * variance)
    )
    return {
        "test_nll": float(np.mean(nll)),
        "one_step_rmse": float(np.sqrt(np.mean((dx - mean) ** 2))),
    }


def simulate_rollout_numpy(predictor, spec, x0, noise):
    dt = float(spec["dt"])
    sqrt_dt = math.sqrt(dt)
    n_steps, n_paths, d = noise.shape
    x = np.asarray(x0, dtype=np.float64).copy()
    paths = np.empty((n_paths, n_steps + 1, d), dtype=np.float64)
    paths[:, 0, :] = x
    diverged = np.zeros(n_paths, dtype=bool)
    state_limit = float(CONFIG["ROLLOUT_STATE_ABS_LIMIT"])

    for step_index in range(n_steps):
        t = np.full(n_paths, step_index * dt, dtype=np.float64)
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            f, sigma = predictor["predict_f_sigma"](t, x)
            x_next = x + f * dt + sigma * sqrt_dt * noise[step_index]

        bad = (
            ~np.isfinite(x_next).all(axis=1)
            | (np.max(np.abs(x_next), axis=1) > state_limit)
        )
        diverged |= bad

        # Keep metrics finite while recording divergence explicitly.
        x_next = np.nan_to_num(
            x_next,
            nan=0.0,
            posinf=state_limit,
            neginf=-state_limit,
        )
        x_next = np.clip(x_next, -state_limit, state_limit)
        x = x_next
        paths[:, step_index + 1, :] = x

    return paths, diverged


def rollout_metrics(predictor, truth_predictor, spec, rollout_x0, rollout_noise):
    truth_paths, truth_diverged = simulate_rollout_numpy(
        truth_predictor,
        spec,
        rollout_x0,
        rollout_noise,
    )
    predicted_paths, predicted_diverged = simulate_rollout_numpy(
        predictor,
        spec,
        rollout_x0,
        rollout_noise,
    )

    if truth_diverged.any():
        raise RuntimeError("Ground-truth rollout diverged under the configured benchmark.")

    squared_error = (predicted_paths - truth_paths) ** 2
    mse = float(np.mean(squared_error))
    divergence_fraction = float(np.mean(predicted_diverged))
    mse += divergence_fraction * float(CONFIG["ROLLOUT_DIVERGENCE_PENALTY"])

    true_terminal = truth_paths[:, -1, :]
    predicted_terminal = predicted_paths[:, -1, :]
    mean_error = float(
        np.linalg.norm(
            np.mean(predicted_terminal, axis=0) - np.mean(true_terminal, axis=0)
        )
    )

    if predicted_terminal.shape[0] >= 2:
        true_cov = np.atleast_2d(np.cov(true_terminal, rowvar=False))
        predicted_cov = np.atleast_2d(np.cov(predicted_terminal, rowvar=False))
        covariance_error = float(np.linalg.norm(predicted_cov - true_cov, ord="fro"))
    else:
        covariance_error = np.nan

    return {
        "rollout_mse": mse,
        "rollout_rmse": float(math.sqrt(max(mse, 0.0))),
        "terminal_mean_l2": mean_error,
        "terminal_cov_fro": covariance_error,
        "rollout_divergence_fraction": divergence_fraction,
    }


def sindy_sparsity_metrics(model):
    drift_nnz = int(
        np.count_nonzero(
            np.abs(model.drift_coefficients) >= float(CONFIG["SINDY_COEF_ZERO_TOL"])
        )
    )
    diffusion_nnz = int(
        np.count_nonzero(
            np.abs(model.q_coefficients) >= float(CONFIG["SINDY_COEF_ZERO_TOL"])
        )
    )
    total_candidates = int(
        model.drift_coefficients.size + model.q_coefficients.size
    )
    total_nnz = drift_nnz + diffusion_nnz
    number_of_equations = model.drift_coefficients.shape[0] + model.q_coefficients.shape[0]

    return {
        "drift_nnz": drift_nnz,
        "diffusion_nnz": diffusion_nnz,
        "total_candidate_coefficients": total_candidates,
        "active_fraction": float(total_nnz / max(total_candidates, 1)),
        "mean_terms_per_equation": float(total_nnz / max(number_of_equations, 1)),
    }


def evaluate_method_record(
    *,
    benchmark_name,
    regime,
    seed,
    data_bundle,
    evaluation_bundle,
    method_name,
    library_label,
    predictor,
    truth_predictor,
    fit_seconds,
    total_seconds,
    stage_i_seconds=np.nan,
    stage_ii_seconds=np.nan,
    stage_iii_seconds=np.nan,
    sindy_model=None,
    coefficient_csv="",
    cv_path_csv="",
):
    spec = BENCHMARKS[benchmark_name]

    box = coefficient_metrics(
        predictor,
        truth_predictor,
        evaluation_bundle["t_box"],
        evaluation_bundle["x_box"],
    )
    support = coefficient_metrics(
        predictor,
        truth_predictor,
        evaluation_bundle["t_support"],
        evaluation_bundle["x_support"],
    )
    one_step = one_step_metrics(predictor, data_bundle["test_table"])
    rollout = rollout_metrics(
        predictor,
        truth_predictor,
        spec,
        evaluation_bundle["rollout_x0"],
        evaluation_bundle["rollout_noise"],
    )

    if sindy_model is None:
        sparsity = {
            "drift_nnz": np.nan,
            "diffusion_nnz": np.nan,
            "total_candidate_coefficients": np.nan,
            "active_fraction": np.nan,
            "mean_terms_per_equation": np.nan,
        }
    else:
        sparsity = sindy_sparsity_metrics(sindy_model)

    record = {
        "seed": int(seed),
        "benchmark": benchmark_name,
        "benchmark_display": spec["display_name"],
        "dimension": int(spec["d"]),
        "regime": regime,
        "method": method_name,
        "library": library_label,
        "n_train_trajectories": int(data_bundle["n_trajectories"]),
        "n_train_transitions": int(table_length(data_bundle["train_table"])),
        "n_test_transitions": int(table_length(data_bundle["test_table"])),

        "drift_mse_box": box["drift_mse"],
        "sigma_mse_box": box["sigma_mse"],
        "q_mse_box": box["q_mse"],
        "drift_mse_support": support["drift_mse"],
        "sigma_mse_support": support["sigma_mse"],
        "q_mse_support": support["q_mse"],

        "rollout_mse": rollout["rollout_mse"],
        "rollout_rmse": rollout["rollout_rmse"],
        "terminal_mean_l2": rollout["terminal_mean_l2"],
        "terminal_cov_fro": rollout["terminal_cov_fro"],
        "rollout_divergence_fraction": rollout["rollout_divergence_fraction"],

        "test_nll": one_step["test_nll"],
        "one_step_rmse": one_step["one_step_rmse"],

        "fit_seconds": float(fit_seconds),
        "total_seconds": float(total_seconds),
        "stage_i_seconds": float(stage_i_seconds),
        "stage_ii_seconds": float(stage_ii_seconds),
        "stage_iii_seconds": float(stage_iii_seconds),

        "drift_nnz": sparsity["drift_nnz"],
        "diffusion_nnz": sparsity["diffusion_nnz"],
        "total_candidate_coefficients": sparsity["total_candidate_coefficients"],
        "active_fraction": sparsity["active_fraction"],
        "mean_terms_per_equation": sparsity["mean_terms_per_equation"],

        "negative_q_fraction": box["negative_q_fraction"],
        "q_clip_fraction": box["q_clip_fraction"],
        "sindy_coefficient_csv": coefficient_csv,
        "sindy_cv_path_csv": cv_path_csv,
        "execution_mode": CONFIG["EXECUTION_MODE"],
    }

    return record


## Cell 10 — End-to-end case runner, result upsert, full/single/smoke execution

In [ ]:
# =============================================================================
# Cell 10 — End-to-end case runner, result upsert, full/single/smoke execution
# =============================================================================


def run_e5_case(benchmark_name, regime, seed):
    if benchmark_name not in BENCHMARKS:
        raise ValueError(f"Unknown benchmark: {benchmark_name}")
    if regime not in {"efficient", "scarce"}:
        raise ValueError(f"Unknown regime: {regime}")

    spec = BENCHMARKS[benchmark_name]
    print("\n" + "=" * 112)
    print(
        f"E5 CASE | {spec['display_name']} | d={spec['d']} | "
        f"regime={regime} | seed={seed}"
    )
    print("=" * 112)
    print(spec["description"])

    data_bundle = prepare_case_data(benchmark_name, regime, seed)
    print(
        f"Trajectories: {data_bundle['n_trajectories']} | "
        f"train transitions: {table_length(data_bundle['train_table'])} | "
        f"test transitions: {table_length(data_bundle['test_table'])}"
    )

    # ---------------------------------------------------------------------
    # Neural baseline and Gaeta--Lie
    # ---------------------------------------------------------------------
    baseline = train_vanilla_neural_sde(
        spec,
        data_bundle["train_table"],
        data_bundle["test_table"],
        seed,
    )

    basis = build_symmetry_basis(spec)
    discovery = discover_symmetries(
        baseline["model"],
        baseline["params"],
        spec,
        basis,
        baseline["train_batch"],
        seed,
    )
    refinement = refine_gaeta_lie(
        baseline["model"],
        baseline["params"],
        spec,
        discovery["W"],
        basis,
        baseline["train_batch"],
        baseline["test_batch"],
        seed,
    )

    # ---------------------------------------------------------------------
    # Stochastic SINDy standard and oracle-enriched libraries
    # ---------------------------------------------------------------------
    sindy_standard = fit_stochastic_sindy(
        benchmark_name,
        "standard",
        data_bundle["train_table"],
        seed,
    )
    print_sindy_equations(sindy_standard)
    standard_coefficient_csv, standard_cv_csv = save_sindy_diagnostics(
        sindy_standard,
        benchmark_name,
        regime,
        seed,
    )

    sindy_oracle = fit_stochastic_sindy(
        benchmark_name,
        "oracle",
        data_bundle["train_table"],
        seed,
    )
    print_sindy_equations(sindy_oracle)
    oracle_coefficient_csv, oracle_cv_csv = save_sindy_diagnostics(
        sindy_oracle,
        benchmark_name,
        regime,
        seed,
    )

    # ---------------------------------------------------------------------
    # Shared evaluation points, initial conditions, and Brownian increments
    # ---------------------------------------------------------------------
    evaluation_bundle = make_evaluation_bundle(
        spec,
        benchmark_name,
        regime,
        seed,
        data_bundle["test_table"],
    )
    truth_predictor = make_true_predictor(spec)

    baseline_predictor = make_neural_predictor(
        baseline["model"],
        baseline["params"],
        spec,
    )
    refined_predictor = make_neural_predictor(
        baseline["model"],
        refinement["params"],
        spec,
    )
    standard_predictor = make_sindy_predictor(sindy_standard)
    oracle_predictor = make_sindy_predictor(sindy_oracle)

    records = []

    records.append(
        evaluate_method_record(
            benchmark_name=benchmark_name,
            regime=regime,
            seed=seed,
            data_bundle=data_bundle,
            evaluation_bundle=evaluation_bundle,
            method_name="Vanilla neural SDE",
            library_label="neural coefficients; no coefficient library",
            predictor=baseline_predictor,
            truth_predictor=truth_predictor,
            fit_seconds=baseline["seconds"],
            total_seconds=baseline["seconds"],
            stage_i_seconds=baseline["seconds"],
        )
    )

    gaeta_lie_total = baseline["seconds"] + discovery["seconds"] + refinement["seconds"]
    records.append(
        evaluate_method_record(
            benchmark_name=benchmark_name,
            regime=regime,
            seed=seed,
            data_bundle=data_bundle,
            evaluation_bundle=evaluation_bundle,
            method_name="Gaeta-Lie",
            library_label=f"generator library R={len(basis)}; neural coefficients",
            predictor=refined_predictor,
            truth_predictor=truth_predictor,
            fit_seconds=discovery["seconds"] + refinement["seconds"],
            total_seconds=gaeta_lie_total,
            stage_i_seconds=baseline["seconds"],
            stage_ii_seconds=discovery["seconds"],
            stage_iii_seconds=refinement["seconds"],
        )
    )

    records.append(
        evaluate_method_record(
            benchmark_name=benchmark_name,
            regime=regime,
            seed=seed,
            data_bundle=data_bundle,
            evaluation_bundle=evaluation_bundle,
            method_name="Stochastic SINDy (standard)",
            library_label=sindy_standard.library.name,
            predictor=standard_predictor,
            truth_predictor=truth_predictor,
            fit_seconds=sindy_standard.fit_seconds,
            total_seconds=sindy_standard.fit_seconds,
            sindy_model=sindy_standard,
            coefficient_csv=standard_coefficient_csv,
            cv_path_csv=standard_cv_csv,
        )
    )

    records.append(
        evaluate_method_record(
            benchmark_name=benchmark_name,
            regime=regime,
            seed=seed,
            data_bundle=data_bundle,
            evaluation_bundle=evaluation_bundle,
            method_name="Stochastic SINDy (oracle)",
            library_label=sindy_oracle.library.name,
            predictor=oracle_predictor,
            truth_predictor=truth_predictor,
            fit_seconds=sindy_oracle.fit_seconds,
            total_seconds=sindy_oracle.fit_seconds,
            sindy_model=sindy_oracle,
            coefficient_csv=oracle_coefficient_csv,
            cv_path_csv=oracle_cv_csv,
        )
    )

    case_table = pd.DataFrame(records)
    print("\nCase-level headline metrics:")
    print(
        case_table[
            [
                "method",
                "drift_mse_box",
                "sigma_mse_box",
                "q_mse_box",
                "rollout_rmse",
                "test_nll",
                "total_seconds",
                "active_fraction",
                "negative_q_fraction",
            ]
        ].to_string(index=False)
    )
    return records


def load_existing_results():
    if (
        bool(CONFIG["RESUME_FROM_EXISTING_CSV"])
        and LONG_RESULTS_PATH.exists()
    ):
        existing = pd.read_csv(LONG_RESULTS_PATH)
        print(f"Loaded {len(existing)} existing E5 records from {LONG_RESULTS_PATH}.")
        return existing.to_dict(orient="records")
    return []


def upsert_result_records(existing_records, new_records):
    key_columns = ["seed", "benchmark", "regime", "method"]
    combined = pd.DataFrame(list(existing_records) + list(new_records))
    if combined.empty:
        return []
    combined = combined.drop_duplicates(subset=key_columns, keep="last")
    combined = combined.sort_values(
        ["dimension", "benchmark", "regime", "seed", "method"]
    ).reset_index(drop=True)
    combined.to_csv(LONG_RESULTS_PATH, index=False)
    return combined.to_dict(orient="records")


def completed_case_keys(records):
    if not records:
        return set()
    table = pd.DataFrame(records)
    completed = set()
    for key, group in table.groupby(["benchmark", "regime", "seed"]):
        if set(METHOD_ORDER).issubset(set(group["method"])):
            completed.add((str(key[0]), str(key[1]), int(key[2])))
    return completed


def build_run_plan():
    mode = CONFIG["EXECUTION_MODE"]
    if mode in {"single", "smoke"}:
        return [
            (
                CONFIG["ACTIVE_BENCHMARK"],
                CONFIG["ACTIVE_REGIME"],
                int(CONFIG["ACTIVE_SEED"]),
            )
        ]
    if mode == "full":
        benchmarks = list(CONFIG["E5_BENCHMARKS"])
        if bool(CONFIG["INCLUDE_OPTIONAL_50D"]):
            benchmarks.append(CONFIG["OPTIONAL_50D_BENCHMARK"])
        return [
            (benchmark, regime, int(seed))
            for benchmark in benchmarks
            for regime in CONFIG["E5_REGIMES"]
            for seed in CONFIG["E5_SEEDS"]
        ]
    raise ValueError(
        "EXECUTION_MODE must be one of {'smoke', 'single', 'full'}."
    )


E5_RESULTS = load_existing_results()

if bool(CONFIG["RUN_E5_NOW"]):
    plan = build_run_plan()
    completed = completed_case_keys(E5_RESULTS)
    print(f"Planned E5 cases: {len(plan)}")

    for benchmark_name, regime, seed in plan:
        case_key = (benchmark_name, regime, int(seed))
        if bool(CONFIG["SKIP_COMPLETED_CASES"]) and case_key in completed:
            print("Skipping completed case:", case_key)
            continue

        new_records = run_e5_case(benchmark_name, regime, seed)
        E5_RESULTS = upsert_result_records(E5_RESULTS, new_records)
        completed = completed_case_keys(E5_RESULTS)
        print(f"Saved cumulative long-form results to {LONG_RESULTS_PATH}")
else:
    print("RUN_E5_NOW=False: no experiment was launched.")



## Cell 11 — Automatic E5 audit, complete numerical report, paired comparisons, CSV export

In [ ]:
# =============================================================================
# Cell 11 — Automatic E5 audit, complete numerical report, paired comparisons, CSV export
# =============================================================================


if "E5_RESULTS" not in globals() or len(E5_RESULTS) == 0:
    raise RuntimeError(
        "E5_RESULTS is empty. Run Cell 10 before executing this final cell."
    )

results = pd.DataFrame(E5_RESULTS).copy()

required_columns = [
    "seed",
    "benchmark",
    "dimension",
    "regime",
    "method",
    "library",
    "n_train_trajectories",
    "n_train_transitions",
    "drift_mse_box",
    "sigma_mse_box",
    "q_mse_box",
    "rollout_mse",
    "rollout_rmse",
    "test_nll",
    "fit_seconds",
    "total_seconds",
]
missing_columns = [column for column in required_columns if column not in results.columns]
if missing_columns:
    raise KeyError(
        "E5 result records are missing required columns: "
        + ", ".join(missing_columns)
    )

optional_columns = [
    "benchmark_display",
    "n_test_transitions",
    "drift_mse_support",
    "sigma_mse_support",
    "q_mse_support",
    "terminal_mean_l2",
    "terminal_cov_fro",
    "rollout_divergence_fraction",
    "one_step_rmse",
    "stage_i_seconds",
    "stage_ii_seconds",
    "stage_iii_seconds",
    "drift_nnz",
    "diffusion_nnz",
    "total_candidate_coefficients",
    "active_fraction",
    "mean_terms_per_equation",
    "negative_q_fraction",
    "q_clip_fraction",
    "sindy_coefficient_csv",
    "sindy_cv_path_csv",
    "execution_mode",
]
for column in optional_columns:
    if column not in results.columns:
        results[column] = np.nan

numeric_columns = [
    "seed",
    "dimension",
    "n_train_trajectories",
    "n_train_transitions",
    "n_test_transitions",
    "drift_mse_box",
    "sigma_mse_box",
    "q_mse_box",
    "drift_mse_support",
    "sigma_mse_support",
    "q_mse_support",
    "rollout_mse",
    "rollout_rmse",
    "terminal_mean_l2",
    "terminal_cov_fro",
    "rollout_divergence_fraction",
    "test_nll",
    "one_step_rmse",
    "fit_seconds",
    "total_seconds",
    "stage_i_seconds",
    "stage_ii_seconds",
    "stage_iii_seconds",
    "drift_nnz",
    "diffusion_nnz",
    "total_candidate_coefficients",
    "active_fraction",
    "mean_terms_per_equation",
    "negative_q_fraction",
    "q_clip_fraction",
]
for column in numeric_columns:
    results[column] = pd.to_numeric(results[column], errors="coerce")

results["method_rank"] = (
    results["method"]
    .map({method: index for index, method in enumerate(METHOD_ORDER)})
    .fillna(999)
    .astype(int)
)
results = results.sort_values(
    ["dimension", "benchmark", "regime", "seed", "method_rank"]
).reset_index(drop=True)


def pass_fail(flag):
    return "PASS" if bool(flag) else "FAIL"


def mean_std_text(mean_value, std_value, scientific=True):
    if pd.isna(mean_value):
        return "N/A"
    if pd.isna(std_value):
        std_value = 0.0
    if scientific:
        return f"{float(mean_value):.4e} +/- {float(std_value):.2e}"
    return f"{float(mean_value):.3f} +/- {float(std_value):.3f}"


# -----------------------------------------------------------------------------
# A. Design-completeness audit
# -----------------------------------------------------------------------------
benchmark_definitions = results[["benchmark", "dimension"]].drop_duplicates()
n_1d = benchmark_definitions.loc[
    benchmark_definitions["dimension"] == 1, "benchmark"
].nunique()
n_3d = benchmark_definitions.loc[
    benchmark_definitions["dimension"] == 3, "benchmark"
].nunique()
n_20d = benchmark_definitions.loc[
    benchmark_definitions["dimension"] == 20, "benchmark"
].nunique()

regimes_per_benchmark = results.groupby("benchmark")["regime"].agg(set)
has_both_regimes = bool(
    len(regimes_per_benchmark)
    and regimes_per_benchmark.map(
        lambda value: {"efficient", "scarce"}.issubset(value)
    ).all()
)

methods_per_case = results.groupby(
    ["seed", "benchmark", "regime"],
    dropna=False,
)["method"].agg(set)
has_all_methods = bool(
    len(methods_per_case)
    and methods_per_case.map(
        lambda value: set(METHOD_ORDER).issubset(value)
    ).all()
)

seeds_per_block = results.groupby(
    ["benchmark", "regime", "method"],
    dropna=False,
)["seed"].nunique()
minimum_seed_count = int(seeds_per_block.min()) if len(seeds_per_block) else 0
has_five_seeds = minimum_seed_count >= 5

sindy_rows = results[results["method"].str.contains("SINDy", regex=False)]
sindy_outputs_complete = bool(
    len(sindy_rows)
    and sindy_rows[
        [
            "drift_nnz",
            "diffusion_nnz",
            "total_candidate_coefficients",
            "active_fraction",
            "negative_q_fraction",
            "q_clip_fraction",
        ]
    ].notna().all(axis=None)
)

core_metrics_finite = bool(
    np.isfinite(
        results[
            [
                "drift_mse_box",
                "sigma_mse_box",
                "q_mse_box",
                "rollout_rmse",
                "test_nll",
                "total_seconds",
            ]
        ].to_numpy(dtype=float)
    ).all()
)

design_checks = [
    ("At least two 1D benchmarks", n_1d >= 2, f"found {n_1d}"),
    ("At least two 3D benchmarks", n_3d >= 2, f"found {n_3d}"),
    ("At least one 20D benchmark", n_20d >= 1, f"found {n_20d}"),
    (
        "Efficient and scarce regimes for every benchmark",
        has_both_regimes,
        "",
    ),
    (
        "All four methods in every completed case",
        has_all_methods,
        "",
    ),
    (
        "Five seeds per benchmark/regime/method",
        has_five_seeds,
        f"minimum found {minimum_seed_count}",
    ),
    (
        "SINDy sparsity and positivity diagnostics present",
        sindy_outputs_complete,
        "",
    ),
    ("All headline numerical metrics finite", core_metrics_finite, ""),
]
overall_design_complete = all(flag for _, flag, _ in design_checks)

print("=" * 120)
print("E5 STOCHASTIC SINDY COEFFICIENT-IDENTIFICATION BASELINE")
print("AUTOMATIC DESIGN-COMPLETENESS AUDIT")
print("=" * 120)
for label, flag, note in design_checks:
    suffix = f" ({note})" if note else ""
    print(f"[{pass_fail(flag)}] {label}{suffix}")
print("-" * 120)
print("OVERALL E5 DESIGN STATUS:", pass_fail(overall_design_complete))
if CONFIG["EXECUTION_MODE"] == "smoke":
    print(
        "WARNING: smoke-mode results use truncated training and libraries and "
        "must not be used for reported results."
    )


# -----------------------------------------------------------------------------
# B. Every run-level numerical result
# -----------------------------------------------------------------------------
run_level_columns = [
    "seed",
    "benchmark",
    "dimension",
    "regime",
    "n_train_trajectories",
    "n_train_transitions",
    "method",
    "library",
    "drift_mse_box",
    "sigma_mse_box",
    "q_mse_box",
    "drift_mse_support",
    "sigma_mse_support",
    "rollout_rmse",
    "rollout_divergence_fraction",
    "test_nll",
    "one_step_rmse",
    "fit_seconds",
    "total_seconds",
    "drift_nnz",
    "diffusion_nnz",
    "active_fraction",
    "negative_q_fraction",
    "q_clip_fraction",
]

print("\n" + "=" * 120)
print("RUN-LEVEL NUMERICAL RESULTS")
print("=" * 120)
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 260,
    "display.float_format", lambda value: f"{value:.6e}",
):
    print(results[run_level_columns].to_string(index=False))


# -----------------------------------------------------------------------------
# C. Mean +/- standard deviation over seeds
# -----------------------------------------------------------------------------
summary_metrics = [
    "drift_mse_box",
    "sigma_mse_box",
    "q_mse_box",
    "drift_mse_support",
    "sigma_mse_support",
    "q_mse_support",
    "rollout_mse",
    "rollout_rmse",
    "terminal_mean_l2",
    "terminal_cov_fro",
    "rollout_divergence_fraction",
    "test_nll",
    "one_step_rmse",
    "fit_seconds",
    "total_seconds",
    "stage_i_seconds",
    "stage_ii_seconds",
    "stage_iii_seconds",
    "drift_nnz",
    "diffusion_nnz",
    "total_candidate_coefficients",
    "active_fraction",
    "mean_terms_per_equation",
    "negative_q_fraction",
    "q_clip_fraction",
]
summary_keys = ["benchmark", "dimension", "regime", "method", "library"]
summary = (
    results.groupby(summary_keys, dropna=False)[summary_metrics]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary.columns = [
    "__".join(str(part) for part in column if str(part) != "")
    if isinstance(column, tuple)
    else str(column)
    for column in summary.columns
]
for metric in summary_metrics:
    std_column = f"{metric}__std"
    if std_column in summary.columns:
        summary[std_column] = summary[std_column].fillna(0.0)
summary["method_rank"] = (
    summary["method"]
    .map({method: index for index, method in enumerate(METHOD_ORDER)})
    .fillna(999)
    .astype(int)
)
summary = summary.sort_values(
    ["dimension", "benchmark", "regime", "method_rank"]
).reset_index(drop=True)

headline_table = summary[
    ["benchmark", "dimension", "regime", "method", "library"]
].copy()
headline_specs = [
    ("drift_mse_box", "drift MSE"),
    ("sigma_mse_box", "sigma MSE"),
    ("q_mse_box", "q MSE"),
    ("rollout_rmse", "rollout RMSE"),
    ("test_nll", "test NLL"),
    ("total_seconds", "total seconds"),
]
for metric, label in headline_specs:
    headline_table[label] = [
        mean_std_text(
            mean_value,
            std_value,
            scientific=(metric != "total_seconds"),
        )
        for mean_value, std_value in zip(
            summary[f"{metric}__mean"],
            summary[f"{metric}__std"],
        )
    ]

print("\n" + "=" * 120)
print("BENCHMARK-LEVEL SUMMARY: MEAN +/- STANDARD DEVIATION OVER SEEDS")
print("=" * 120)
print(headline_table.to_string(index=False))


# -----------------------------------------------------------------------------
# D. Paired percentage reductions
# Positive = challenger has lower error/runtime than reference.
# -----------------------------------------------------------------------------
comparison_specs = [
    ("Vanilla neural SDE", "Gaeta-Lie"),
    ("Vanilla neural SDE", "Stochastic SINDy (standard)"),
    ("Vanilla neural SDE", "Stochastic SINDy (oracle)"),
    ("Gaeta-Lie", "Stochastic SINDy (standard)"),
    ("Gaeta-Lie", "Stochastic SINDy (oracle)"),
]
comparison_metrics = [
    "drift_mse_box",
    "sigma_mse_box",
    "q_mse_box",
    "rollout_rmse",
    "test_nll",
    "total_seconds",
]
pair_index = ["seed", "benchmark", "dimension", "regime"]
comparison_records = []

for metric in comparison_metrics:
    pivot = results.pivot_table(
        index=pair_index,
        columns="method",
        values=metric,
        aggfunc="first",
    )
    for reference, challenger in comparison_specs:
        if reference not in pivot.columns or challenger not in pivot.columns:
            continue
        paired = pivot[[reference, challenger]].dropna()
        if paired.empty:
            continue
        reduction = 100.0 * (paired[reference] - paired[challenger]) / (
            np.abs(paired[reference]) + 1.0e-12
        )
        for index_values, value in reduction.items():
            seed, benchmark, dimension, regime = index_values
            comparison_records.append(
                {
                    "seed": int(seed),
                    "benchmark": benchmark,
                    "dimension": int(dimension),
                    "regime": regime,
                    "metric": metric,
                    "reference": reference,
                    "challenger": challenger,
                    "reduction_percent": float(value),
                }
            )

comparison_details = pd.DataFrame(comparison_records)
if len(comparison_details):
    comparison_summary = (
        comparison_details.groupby(
            ["regime", "metric", "reference", "challenger"],
            dropna=False,
        )["reduction_percent"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    comparison_summary["std"] = comparison_summary["std"].fillna(0.0)
    comparison_summary["mean +/- std (%)"] = [
        mean_std_text(mean_value, std_value, scientific=False)
        for mean_value, std_value in zip(
            comparison_summary["mean"],
            comparison_summary["std"],
        )
    ]
    comparison_summary = comparison_summary.sort_values(
        ["regime", "metric", "reference", "challenger"]
    )

    print("\n" + "=" * 120)
    print("PAIRED PERCENTAGE REDUCTIONS: POSITIVE = CHALLENGER IS LOWER")
    print("=" * 120)
    print(
        comparison_summary[
            [
                "regime",
                "metric",
                "reference",
                "challenger",
                "mean +/- std (%)",
                "count",
            ]
        ].to_string(index=False)
    )
else:
    comparison_summary = pd.DataFrame()
    print("\nNo complete paired comparisons are available yet.")


# -----------------------------------------------------------------------------
# E. SINDy sparsity, CV-selected model sizes, and diffusion positivity
# -----------------------------------------------------------------------------
if len(sindy_rows):
    sparsity_metrics = [
        "drift_nnz",
        "diffusion_nnz",
        "total_candidate_coefficients",
        "active_fraction",
        "mean_terms_per_equation",
        "negative_q_fraction",
        "q_clip_fraction",
        "fit_seconds",
        "total_seconds",
    ]
    sparsity_summary = (
        sindy_rows.groupby(
            ["benchmark", "dimension", "regime", "method", "library"],
            dropna=False,
        )[sparsity_metrics]
        .agg(["mean", "std"])
        .reset_index()
    )
    sparsity_summary.columns = [
        "__".join(str(part) for part in column if str(part) != "")
        if isinstance(column, tuple)
        else str(column)
        for column in sparsity_summary.columns
    ]
    for metric in sparsity_metrics:
        sparsity_summary[f"{metric}__std"] = sparsity_summary[
            f"{metric}__std"
        ].fillna(0.0)

    sparsity_table = sparsity_summary[
        ["benchmark", "dimension", "regime", "method", "library"]
    ].copy()
    for metric, label in [
        ("drift_nnz", "drift nnz"),
        ("diffusion_nnz", "q nnz"),
        ("active_fraction", "active fraction"),
        ("mean_terms_per_equation", "terms/equation"),
        ("negative_q_fraction", "raw q<0 fraction"),
        ("q_clip_fraction", "q clipped fraction"),
        ("total_seconds", "total seconds"),
    ]:
        sparsity_table[label] = [
            mean_std_text(mean_value, std_value, scientific=False)
            for mean_value, std_value in zip(
                sparsity_summary[f"{metric}__mean"],
                sparsity_summary[f"{metric}__std"],
            )
        ]

    print("\n" + "=" * 120)
    print("SINDY SPARSITY, RUNTIME, AND DIFFUSION-POSITIVITY DIAGNOSTICS")
    print("=" * 120)
    print(sparsity_table.to_string(index=False))
else:
    sparsity_summary = pd.DataFrame()


# -----------------------------------------------------------------------------
# F. Pooled summary across benchmarks
# -----------------------------------------------------------------------------
overall_metrics = [
    "drift_mse_box",
    "sigma_mse_box",
    "q_mse_box",
    "rollout_rmse",
    "test_nll",
    "total_seconds",
]
overall = (
    results.groupby(["regime", "method"], dropna=False)[overall_metrics]
    .agg(["mean", "std", "count"])
    .reset_index()
)
overall.columns = [
    "__".join(str(part) for part in column if str(part) != "")
    if isinstance(column, tuple)
    else str(column)
    for column in overall.columns
]
overall["method_rank"] = (
    overall["method"]
    .map({method: index for index, method in enumerate(METHOD_ORDER)})
    .fillna(999)
    .astype(int)
)
overall = overall.sort_values(["regime", "method_rank"])

overall_table = overall[["regime", "method"]].copy()
for metric, label in [
    ("drift_mse_box", "drift MSE"),
    ("sigma_mse_box", "sigma MSE"),
    ("q_mse_box", "q MSE"),
    ("rollout_rmse", "rollout RMSE"),
    ("test_nll", "test NLL"),
    ("total_seconds", "total seconds"),
]:
    overall_table[label] = [
        mean_std_text(
            mean_value,
            0.0 if pd.isna(std_value) else std_value,
            scientific=(metric != "total_seconds"),
        )
        for mean_value, std_value in zip(
            overall[f"{metric}__mean"],
            overall[f"{metric}__std"],
        )
    ]

print("\n" + "=" * 120)
print("OVERALL E5 SUMMARY POOLED ACROSS BENCHMARKS")
print("=" * 120)
print(overall_table.to_string(index=False))


# -----------------------------------------------------------------------------
# G. Automatic interpretation counters
# -----------------------------------------------------------------------------
def paired_win_count(metric, challenger, reference):
    pivot = results.pivot_table(
        index=["seed", "benchmark", "regime"],
        columns="method",
        values=metric,
        aggfunc="first",
    )
    if challenger not in pivot.columns or reference not in pivot.columns:
        return 0, 0
    paired = pivot[[challenger, reference]].dropna()
    wins = int((paired[challenger] < paired[reference]).sum())
    return wins, int(len(paired))


print("\n" + "=" * 120)
print("AUTOMATIC PAIRED WIN COUNTS")
print("=" * 120)
for metric in ["drift_mse_box", "sigma_mse_box", "rollout_rmse"]:
    for challenger, reference in [
        ("Gaeta-Lie", "Stochastic SINDy (standard)"),
        ("Gaeta-Lie", "Stochastic SINDy (oracle)"),
        ("Stochastic SINDy (oracle)", "Stochastic SINDy (standard)"),
    ]:
        wins, total = paired_win_count(metric, challenger, reference)
        print(f"{metric}: {challenger} beats {reference} in {wins}/{total} paired runs")


# -----------------------------------------------------------------------------
# H. Save all machine-readable tables
# -----------------------------------------------------------------------------
summary_path = RESULTS_DIR / "E5_results_summary.csv"
comparison_details_path = RESULTS_DIR / "E5_paired_reductions.csv"
comparison_summary_path = RESULTS_DIR / "E5_paired_reductions_summary.csv"
overall_path = RESULTS_DIR / "E5_overall_summary.csv"
sparsity_path = RESULTS_DIR / "E5_sindy_sparsity_summary.csv"
audit_path = RESULTS_DIR / "E5_design_audit.json"

results.drop(columns=["method_rank"]).to_csv(LONG_RESULTS_PATH, index=False)
summary.drop(columns=["method_rank"]).to_csv(summary_path, index=False)
comparison_details.to_csv(comparison_details_path, index=False)
comparison_summary.to_csv(comparison_summary_path, index=False)
overall.drop(columns=["method_rank"]).to_csv(overall_path, index=False)
sparsity_summary.to_csv(sparsity_path, index=False)

with open(audit_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            "overall_design_complete": bool(overall_design_complete),
            "checks": [
                {"name": label, "passed": bool(flag), "note": note}
                for label, flag, note in design_checks
            ],
            "execution_mode": CONFIG["EXECUTION_MODE"],
            "number_of_records": int(len(results)),
        },
        file,
        indent=2,
    )

print("\n" + "=" * 120)
print("SAVED E5 OUTPUT FILES")
print("=" * 120)
for path in [
    LONG_RESULTS_PATH,
    summary_path,
    comparison_details_path,
    comparison_summary_path,
    overall_path,
    sparsity_path,
    audit_path,
]:
    print(path)

print(
    "\nInterpretation: coefficient errors, rollout errors, NLL, and runtime "
    "are lower-is-better. Positive paired percentage reduction means that "
    "the challenger improved over the reference."
)
